In [1]:
# Import der für die Analyse und Visualisierung benötigten Bibliotheken
import pandas as pd
import plotly.express as px

In [2]:
# Import für die API-Schnittstelle
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests

# Einführung

## Thema

Was sammelt ein Weltmuseum? Eine datenanalytische Untersuchung der Sammlungsbestände des Metropolitan Museum of Art mit einer Fallstudie zum iranischen Kulturerbe

## Schritte der Datenanalyse:

1. Fragen & Problemstellung,
2. Daten beschaffen,
3. Daten explorieren (Explorative Datenanalyse),
4. Daten bereinigen,
- (optionale Schritte: Feature Engineering & Predictive Modelling,)
5. Daten analysieren,
6. Daten visualisieren & präsentieren,
7. Handlungsempfehlungen.

## Schritt 1: Frage & Problemstellung

Das Metropolitan Museum of Art (Met) in New York verwaltet eine der größten Museumssammlungen der Welt. Die große Anzahl an Objekten unterschiedlicher Herkunft, Zeiträume und Objektarten bietet die Möglichkeit, die Struktur und Schwerpunkte der Sammlung datenbasiert zu untersuchen. Ziel dieser Analyse ist es, zunächst die Gesamtbestände des Museums zu analysieren und anschließend die iranischen Bestände als Fallstudie vertieft zu untersuchen.

### Übergeordnete Fragestellung

Welche kulturellen, geografischen und zeitlichen Schwerpunkte prägen die Sammlungsbestände des Metropolitan Museum of Art und welche Merkmale kennzeichnen die iranischen Bestände innerhalb dieser Sammlung?

### Leitfragen

1. Welche kulturellen, geografischen und zeitlichen Schwerpunkte lassen sich in den Sammlungsbeständen des Metropolitan Museum of Art erkennen?

2. Welche Rolle spielen die iranischen Bestände innerhalb der Sammlung des Metropolitan Museum of Art?

3. Welche zeitlichen, materiellen und institutionellen Schwerpunkte weisen die iranischen Bestände auf?

## Schritt 2: Daten beschaffen

MetObjects.csv 
Heruntergeladen am 22.09.2026
Link: https://www.kaggle.com/datasets/metmuseum/the-metropolitan-museum-of-art-open-access

– Im Analyszenario: Durch das Museum bereitgestellte Primärdaten aus dem Museumskatalog mit Metadaten. –

In [4]:
# Erster Überblick über den Datensatz anhand des Text-Editors
# Beim Einlesen zunächst keine weiteren Maßnahmen nötig. 
# Bereinigungsschritte erfolgen aufgrund der EDA.

In [5]:
# Einlesen des Datensatzes
df = pd.read_csv("MetObjects.csv")
df.head(3) # Kontrolle

C:\Users\denni\AppData\Local\Temp\ipykernel_23532\3432721099.py:2: DtypeWarning: Columns (0: Culture, 1: Period, 2: Dynasty, 3: Reign, 4: Portfolio, 5: Artist Nationality, 6: Geography Type, 7: City, 8: State, 9: County, 10: Country, 11: Region, 12: Subregion, 13: Locale, 14: Locus, 15: Excavation, 16: River, 17: Rights and Reproduction) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("MetObjects.csv")


,Object Number,Is Highlight,Is Public Domain,Object ID,Department,Object Name,Title,Culture,Period,Dynasty,...,Subregion,Locale,Locus,Excavation,River,Classification,Rights and Reproduction,Link Resource,Metadata Date,Repository
0,1979.486.1,False,False,1,American Decorative Arts,Coin,One-dollar Liberty Head Coin,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Metal,NaN,http://www.metmuseum.org/art/collection/search/1,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
1,1980.264.5,False,False,2,American Decorative Arts,Coin,Ten-dollar Liberty Head Coin,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Metal,NaN,http://www.metmuseum.org/art/collection/search/2,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
2,67.265.9,False,False,3,American Decorative Arts,Coin,Two-and-a-Half Dollar Coin,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Metal,NaN,http://www.metmuseum.org/art/collection/search/3,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"


In [6]:
# Beim Einlesen des Datensatzes tritt eine DtypeWarning auf, da einzelne Spalten gemischte Datentypen enthalten. 
# Dies ist bei umfangreichen Metadatensammlungen nicht ungewöhnlich und wird im Rahmen der Datenbereinigung berücksichtigt.

## Schritt 2.1: Collection-API des Metropolitan Museum of Art

Neben dem Open-Access-Datensatz wird die öffentliche Collection API des Metropolitan Museum of Art genutzt. Die API ermöglicht den automatisierten Abruf von Objektdaten anhand der jeweiligen Object ID und stellt die Daten im JSON-Format bereit. Für dieses Projekt wird die API insbesondere zur Ergänzung des Datensatzes um Bild-URLs verwendet, die für die spätere Visualisierung im Dashboard genutzt werden können. Die API kann ohne Authentifizierung genutzt werden und eignet sich als Beispiel für automatisierte Datenbeschaffung über Programmierschnittstellen.

In [7]:
object_id = df["Object ID"].iloc[0]

url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"

response = requests.get(url)

data = response.json()

In [8]:
# Da die API inhaltlich weitgehend dieselben Metadaten wie der Open-Access-Datensatz bereitstellt, 
# wird der CSV-Datensatz weiterhin als primäre Datenquelle verwendet. 
# Die API dient ergänzend zur automatisierten Anreicherung ausgewählter Objekte 
# um zusätzliche Informationen, insbesondere Bild-URLs.

## Schritt 3: Explorative Datenanalyse

In [9]:
# Überprüfung der Datensatzlänge
len(df) # Der Datensatz umfasst 448203 Zeilen

448203

In [10]:
df.shape # 43 Spalten

(448203, 43)

In [11]:
df.columns # Anzeige aller Spalten

Index(['Object Number', 'Is Highlight', 'Is Public Domain', 'Object ID',
       'Department', 'Object Name', 'Title', 'Culture', 'Period', 'Dynasty',
       'Reign', 'Portfolio', 'Artist Role', 'Artist Prefix',
       'Artist Display Name', 'Artist Display Bio', 'Artist Suffix',
       'Artist Alpha Sort', 'Artist Nationality', 'Artist Begin Date',
       'Artist End Date', 'Object Date', 'Object Begin Date',
       'Object End Date', 'Medium', 'Dimensions', 'Credit Line',
       'Geography Type', 'City', 'State', 'County', 'Country', 'Region',
       'Subregion', 'Locale', 'Locus', 'Excavation', 'River', 'Classification',
       'Rights and Reproduction', 'Link Resource', 'Metadata Date',
       'Repository'],
      dtype='str')

In [12]:
df.dtypes # Anzeige der Datentypen

Object Number                 str
Is Highlight                 bool
Is Public Domain             bool
Object ID                   int64
Department                    str
Object Name                   str
Title                         str
Culture                       str
Period                     object
Dynasty                       str
Reign                         str
Portfolio                     str
Artist Role                   str
Artist Prefix                 str
Artist Display Name           str
Artist Display Bio            str
Artist Suffix                 str
Artist Alpha Sort             str
Artist Nationality            str
Artist Begin Date             str
Artist End Date               str
Object Date                   str
Object Begin Date           int64
Object End Date             int64
Medium                        str
Dimensions                    str
Credit Line                   str
Geography Type                str
City                          str
State         

In [13]:
# Zusammenfassende Kontrolle von SZeilen, palten, Datentypen und NaN-Werten
df.info()

# Bereits hier fallen die vielen NaN-Werte auf!
# Die Datentypen entsprechen weitgehend den Erwartungen eines Sammlungsdatensatzes. 
# Auffällig ist lediglich die Spalte "Period", die als object eingelesen wurde und daher auf mögliche Inkonsistenzen geprüft werden muss.

# Aufgrund des Analyseziels müssen nicht alle Spalten für den Analyse-DF behalten werden. 

<class 'pandas.DataFrame'>
RangeIndex: 448203 entries, 0 to 448202
Data columns (total 43 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Object Number            448203 non-null  str   
 1   Is Highlight             448203 non-null  bool  
 2   Is Public Domain         448203 non-null  bool  
 3   Object ID                448203 non-null  int64 
 4   Department               448203 non-null  str   
 5   Object Name              445568 non-null  str   
 6   Title                    416756 non-null  str   
 7   Culture                  186518 non-null  str   
 8   Period                   71882 non-null   object
 9   Dynasty                  23018 non-null   str   
 10  Reign                    10817 non-null   str   
 11  Portfolio                20370 non-null   str   
 12  Artist Role              259909 non-null  str   
 13  Artist Prefix            88928 non-null   str   
 14  Artist Display Name      261111

In [14]:
# Anzeige der NaN-werte je Spalte
df.isna().sum()

# Die Analyse der fehlenden Werte zeigte deutliche Unterschiede in der Datenqualität der einzelnen Spalten/Attribute. 
# Während zentrale Analysevariablen wie Department, Object Begin Date und Object End Date vollständig vorlagen, 
# weisen zahlreiche geografische und historische Zusatzinformationen einen sehr hohen Anteil fehlender Werte auf.
# Attribute mit unzureichender Datenabdeckung werden in der Datenbereinigung daher aus dem weiteren Analyseprozess ausgeschlossen, 
# um eine belastbare Datengrundlage sicherzustellen.

Object Number                   0
Is Highlight                    0
Is Public Domain                0
Object ID                       0
Department                      0
Object Name                  2635
Title                       31447
Culture                    261685
Period                     376321
Dynasty                    425185
Reign                      437386
Portfolio                  427833
Artist Role                188294
Artist Prefix              359275
Artist Display Name        187092
Artist Display Bio         224139
Artist Suffix              437991
Artist Alpha Sort          187115
Artist Nationality         252071
Artist Begin Date          232969
Artist End Date            235378
Object Date                 15594
Object Begin Date               0
Object End Date                 0
Medium                       8048
Dimensions                  62843
Credit Line                   670
Geography Type             389740
City                       417683
State         

Auffällig ist, dass die Spalte „Link Resource“ keine fehlenden Werte enthält. 
Somit verfügt jedes Objekt über einen zugehörigen Datensatz bzw. einen eindeutigen Verweis innerhalb der digitalen Sammlungsdokumentation des Museums. 
Die Spalte wird für die weitere Analyse nicht benötigt, weist jedoch auf eine vollständige digitale Erschließung der Objekte auf Referenzebene hin.

In [15]:
df.describe().round(2) # Statistische Werte

# Auffällig bei max: Zwei Jahresangaben sind zu einer Zahl zusammengefallen, muss bei der Datenberenigung berücksichtigt werden!

,Object ID,Object Begin Date,Object End Date
count,448203.00,448203.00,448203.00
mean,346272.87,1390.82,1493.45
std,206914.99,27798.16,27787.12
min,1.00,-95000.00,-35000.00
25%,198808.50,1567.00,1600.00
50%,349816.00,1800.00,1850.00
75%,481764.50,1895.00,1910.00
max,750830.00,18591861.00,18591861.00


In [16]:
df.duplicated().sum() # Kontrolle auf direkte Duplikate: keine im Datensatz vorhanden

np.int64(0)

In [17]:
df.duplicated(subset="Object ID").sum() # Prüfung von versteckten Duplikaten über die Object ID: ebenfalls keine im Datensatz vorhanden

np.int64(0)

In [18]:
df.duplicated(subset="Object Number").sum() # Prüfung von versteckten Duplikaten über die Object Number: 2576 versteckte Duplikate

np.int64(2576)

Während anhand der Object ID keine Duplikate festgestellt werden konnten, ergab die Prüfung der Object Number 2576 Mehrfacheinträge. 
Dies deutet darauf hin, dass die Objektnummer nicht als eindeutiger Identifikator verwendet werden kann. Die entsprechenden Datensätze wurden daher näher untersucht.

In [19]:
df[df.duplicated(subset="Object Number", keep=False)].sort_values("Object Number")

# Identische Objektnummern können in unterschiedlichen Museumsabteilungen auftreten können. 
# Die Object Number wird daher nicht als eindeutiger Identifikator verwendet. A
# Als eindeutiger Schlüssel der Analyse dient stattdessen die Object ID.

,Object Number,Is Highlight,Is Public Domain,Object ID,Department,Object Name,Title,Culture,Period,Dynasty,...,Subregion,Locale,Locus,Excavation,River,Classification,Rights and Reproduction,Link Resource,Metadata Date,Repository
66331,01.2.10,False,False,78938,Costume Institute,Fichu,Fichu,French,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
264537,01.2.10,False,True,391695,Drawings and Prints,Print,Musette Player (Le Joueur de Musette),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Prints,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
124314,01.2.4,False,False,212677,European Sculpture and Decorative Arts,Piece,Piece,French,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Textiles-Laces,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
245558,01.2.4,False,False,371969,Drawings and Prints,Print,Marin Mersenne,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Prints,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
124316,01.2.6,False,False,212679,European Sculpture and Decorative Arts,Fragment,Fragment,Belgian,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Textiles-Laces,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405261,X.27,False,False,659590,Drawings and Prints,Print,The Question Settled,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Prints,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
239758,X.40,False,False,365950,Drawings and Prints,Print,"Dedication–Lincoln Center Oct. 26, 1969. Julli...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Prints,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
374758,X.40,False,False,569298,Egyptian Art,"Beads, string",String of Beads,NaN,"New Kingdom, Ramesside",Dynasty 19–20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"
170794,X.650,False,False,271646,Photographs,Photographically illustrated book,The Ruins of Pompeii. A Series of Eighteen Pho...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Books,NaN,http://www.metmuseum.org/art/collection/search...,4/3/2017 8:00:08 AM,"Metropolitan Museum of Art, New York, NY"


In [20]:
# Übersicht über die Departments
df["Department"].value_counts()

Department
Drawings and Prints                          154445
European Sculpture and Decorative Arts        42528
Asian Art                                     36727
Photographs                                   36258
Costume Institute                             33681
Egyptian Art                                  27542
Greek and Roman Art                           17292
Islamic Art                                   15082
Modern and Contemporary Art                   13991
Arms and Armor                                13486
Arts of Africa, Oceania, and the Americas     12427
American Decorative Arts                      12330
Medieval Art                                   7343
Ancient Near Eastern Art                       6185
American Paintings and Sculpture               5451
Musical Instruments                            5303
European Paintings                             2760
The Cloisters                                  2628
Robert Lehman Collection                       2589
T

In [21]:
# Übersicht über die Spalte Culture
df["Culture"].value_counts()

# 7101 unterschiedliche Kultur-Angaben.
# Auffällig bereits hier, dass teilweise Kultur und Ort (z.B. German, Innsbruck) vermischt sind -> Bereinigung!

Culture
American                          22167
French                            18224
Japan                             16374
China                             13844
Italian                            6580
                                  ...  
British, Staffordshire, Fenton        1
German, Innsbruck                     1
Borneo (Sarawak)                      1
Dayak                                 1
French, presumably Paris              1
Name: count, Length: 7101, dtype: int64

In [22]:
# Übersicht über die Länder
df["Country"].value_counts()

# 1035 Länderangaben!
# Aber auch hier auffällig: "or", teilweise "|", teilweise Epochen ("Byzantine Egypt")

Country
Egypt                     30914
United States              8501
Iran                       5886
Peru                       3422
Byzantine Egypt            1673
                          ...  
Byzantium                     1
Iran or Europe                1
Unites States                 1
Turkey or Central Asia        1
Italy|Italy                   1
Name: count, Length: 1035, dtype: int64

In [23]:
# Übersicht über die Region
df["Region"].value_counts()

Region
Upper Egypt, Thebes           11137
Memphite Region                4676
Mesopotamia                    1993
Iran                           1713
Mesoamerica                    1281
                              ...  
Spiro area                        1
Stikine River, Wrangell           1
northwest Vancouver Island        1
Queen Charlotte Island            1
Iban (Dayak)                      1
Name: count, Length: 705, dtype: int64

In [24]:
# Mit Iran-Bezug
df[df["Country"] == "Iran"]["Region"].value_counts()

Region
Khurasan                       13
probably Khurasan               2
Qazvin                          1
Central Asia                    1
Kurdistan                       1
possibly Khurasan               1
Northwestern Iranian region     1
Iran                            1
Western                         1
Name: count, dtype: int64

In [25]:
# Anzeige der Subregion
df["Subregion"].value_counts()

Subregion
Deir el-Bahri             3062
Lisht North               3005
Malqata                   2166
Asasif                    1507
Southern Asasif           1376
                          ... 
Elephantine, Aswan           1
Qaw el-Kebir                 1
El-Rizeiqat                  1
Kerma                        1
Mefkat (Kom Abu Billu)       1
Name: count, Length: 361, dtype: int64

In [26]:
# Mit Iran-Bezug
df[df["Country"] == "Iran"]["Subregion"].value_counts()

Series([], Name: count, dtype: int64)

In [27]:
# Anzeige der Geography Types
df["Geography Type"].value_counts()

Geography Type
From                                          26486
Made in                                       15465
Attributed to                                  7599
Excavated in                                   3335
Probably made in                                957
                                              ...  
Country of Origin|Made for                        1
Country of Origin|Current location|Made in        1
Country of Origin|Probably made in                1
Designed in                                       1
Possibly from|Original from                       1
Name: count, Length: 125, dtype: int64

In [28]:
# Mit Iran-Bezug
df[df["Country"] == "Iran"]["Geography Type"].value_counts()

Geography Type
Excavated in                         3324
Attributed to                        1573
Made in                               491
From                                  164
Found                                 124
Country of Origin                      63
Found/excavated                        53
Probably from                          50
Probably made in                        6
Attributed to|Attributed to             3
Said to be from                         3
Possibly made in                        1
Made in|Attributed to                   1
Possibly from                           1
Excavated in|Attributed to              1
Attributed to|Country of Origin         1
(none assigned)|Country of Origin       1
Name: count, dtype: int64

In [29]:
# Test-DF
df[["Object ID", "Geography Type", "City", "State", "County", "Country"]].sample(10)

,Object ID,Geography Type,City,State,County,Country
244621,370996,NaN,NaN,NaN,NaN,NaN
99801,175856,NaN,NaN,NaN,NaN,NaN
30466,36118,NaN,NaN,NaN,NaN,NaN
88853,125182,NaN,NaN,NaN,NaN,NaN
73483,90377,NaN,NaN,NaN,NaN,NaN
260170,387070,NaN,NaN,NaN,NaN,NaN
252859,379522,NaN,NaN,NaN,NaN,NaN
274099,401480,NaN,NaN,NaN,NaN,NaN
376445,571036,From,NaN,NaN,NaN,Egypt
176854,278382,NaN,NaN,NaN,NaN,NaN


In [30]:
# Erweiterter Iran-Test-DF
df[df["Country"] == "Iran"][["Object ID", "Object Name", "Geography Type", "Country", "Classification", "Medium", "Link Resource"]]

,Object ID,Object Name,Geography Type,Country,Classification,Medium,Link Resource
12834,13965,Rug,Made in,Iran,Textiles,Wool,http://www.metmuseum.org/art/collection/search...
12836,13967,Rug,Made in,Iran,Textiles,Wool,http://www.metmuseum.org/art/collection/search...
12837,13968,Rug,Made in,Iran,Textiles,Wool,http://www.metmuseum.org/art/collection/search...
12838,13969,Rug,Made in,Iran,Textiles,Wool,http://www.metmuseum.org/art/collection/search...
12839,13970,Rug,Possibly made in,Iran,Textiles,Wool,http://www.metmuseum.org/art/collection/search...
...,...,...,...,...,...,...,...
444851,742363,Fragment,Excavated in,Iran,Stucco,Stucco; carved,http://www.metmuseum.org/art/collection/search...
444852,742364,Fragment,Excavated in,Iran,Stucco,"Stucco; carved, painted",http://www.metmuseum.org/art/collection/search...
444853,742365,Fragment,Excavated in,Iran,Stucco,"Stucco; carved, painted",http://www.metmuseum.org/art/collection/search...
444854,742366,Fragment,Excavated in,Iran,Stucco,"Stucco; carved, painted",http://www.metmuseum.org/art/collection/search...


In [31]:
# Anzeige möglicher Iran-bezogener Werte in Spalte Culture (Suchwerte nach Brainstorming)
culture_filter = df["Culture"].str.contains("Iran|Iranian|Persia|Persian|Achaemenid|Parthian|Sasanian|Sassanian|Safavid|Qajar|Seljuq|Timurid|Ilkhanid", regex=True, na=False)

df.loc[culture_filter, "Culture"].unique()


<ArrowStringArray>
[                                                        'Persian',
                                     'Southern Russian or Iranian',
                                              'Turkish or Iranian',
                                              'Iranian or Turkish',
                  'Western Iranian, in the style of Turkman armor',
                                                         'Iranian',
 'grip, Indian; guard, scabbard, blade deco, Turkish; blade, Iran',
                                               'Indian or Iranian',
                                               'Iranian or Indian',
                                                'Persian, Safavid',
                                                'probably Iranian',
                                                  'Persian, Qajar',
 'blade and guard, Turkish; grip, Indian; blade, possibly Iranian',
                  'hilt, Turkish or North African; blade, Iranian',
                      'Hilt a

In [32]:
# Dasselbe über .loc, mit value_counts()
df.loc[df["Culture"].str.contains("Iran|Iranian|Persia|Persian|Achaemenid|Parthian|Sasanian|Sassanian|Safavid|Qajar|Seleucid|Ilkhanid|Timurid|Sogdian|Luristan", regex=True, na=False), "Culture"].value_counts()

Culture
Iran                                               1088
Sasanian                                            917
Achaemenid                                          163
Parthian                                            157
Persian                                             147
                                                   ... 
Syrian (Damascus), Persian, or Egyptian (Cairo)       1
Arabian or Iranian                                    1
Iranian (Persian), Turkey                             1
Iranian (Persia)                                      1
Turkish or Persian                                    1
Name: count, Length: 87, dtype: int64

In [33]:
# Dasselbe über .loc, mit value_counts()
df.loc[df["Culture"].str.contains("Iran|Iranian|Persia|Persian|Achaemenid|Parthian|Sasanian|Sassanian|Safavid|Qajar|Seleucid|Ilkhanid|Timurid|Sogdian|Luristan", regex=True, na=False), "Culture"].value_counts().head(30)

Culture
Iran                            1088
Sasanian                         917
Achaemenid                       163
Parthian                         157
Persian                          147
Sasanian or Islamic               93
Iranian                           61
Babylonian or Achaemenid          60
Seleucid                          46
Iranian (Persian)                 39
Seleucid or Parthian              25
Persian, Qajar                    18
Parthian or Sasanian              12
Parthian (?)                      11
Indo-Persian                      10
Greek or Achaemenid                9
probably Iranian                   7
Graeco-Persian                     7
hilt, Indian; blade, Persian       5
East Greek or Parthian             5
Achaemenid or Seleucid             5
Iranian or Turkish                 4
Turkish with Persian blade         4
Greek, Seleucid                    4
Achaemenid (?)                     4
Sasanian (?)                       4
Qajar                         

In [34]:
df[df["Culture"] == "Iran"]["Country"].value_counts()

Series([], Name: count, dtype: int64)

In [35]:
df.loc[df["Culture"].str.contains("Iran|Iranian|Persia|Persian|Achaemenid|Parthian|Sasanian|Sassanian|Safavid|Qajar", regex=True, na=False)].shape

(2933, 43)

In [36]:
df[df["Country"] == "Persia"].value_counts() # "Persia" kommt als alternative Länderbezeichnung für Iran nicht im Datensatz vor

Series([], Name: count, dtype: int64)

Die explorative Datenanalyse zeigt, dass iranbezogene Einträge in der Spalte Culture ein breites Spektrum kultureller, historischer und geografischer Bezüge umfassen. Neben Einträgen wie "Iran" oder "Iranian" finden sich historische Dynastien und Reiche (z. B. Achaemenid, Parthian, Sasanian), transregionale Kulturformen (z. B. Indo-Persian oder Graeco-Persian) sowie Mehrfachzuordnungen. Dies spiegelt die historische Realität wider, dass der iranisch-persische Kulturraum weit über die Grenzen des heutigen iranischen Staatsgebiets hinauswirkte.

Für die vertiefende Analyse wird daher eine klar definierte Auswahl eindeutig iranisch-persischer Kulturangaben aus der Spalte Culture verwendet. Berücksichtigt werden ausschließlich direkt dem iranisch-persischen Kulturraum zuordenbare Einträge wie "Iran", "Iranian", "Persian", "Achaemenid", "Parthian", "Sasanian" oder "Qajar". Unsichere oder mehrdeutige Zuordnungen (z. B. "Iranian or Turkish" oder "Babylonian or Achaemenid") werden hingegen nicht berücksichtigt. Dadurch kann eine klar abgegrenzte Untersuchungsgruppe gebildet werden, die sowohl Objekte des heutigen Iran als auch Objekte zentraler historischer iranischer Kulturtraditionen umfasst.

In [37]:
df["Metadata Date"].value_counts() 

# Alle Metadatan haben ein einheitliches Datum (2017-04-03) und sind daher für die weitere Analyse nciht wirklich sinnvoll.

Metadata Date
4/3/2017 8:00:08 AM    448203
Name: count, dtype: int64

In [38]:
# Ansicht der Werte des Objekterwerbs
df["Credit Line"].value_counts()

# Die Spalte ist aufgrund dieser Werte für eine Analyse nicht geeignet -> Feature Engineering notwendig!

Credit Line
The Jefferson R. Burdick Collection, Gift of Jefferson R. Burdick                                     41743
Walker Evans Archive, 1994                                                                            10628
The Cesnola Collection, Purchased by subscription, 1874–76                                             5356
Gift of J. Pierpont Morgan, 1917                                                                       4497
Rogers Fund, 1909                                                                                      3648
                                                                                                      ...  
Transferred from Photograph Sales (Gift of Carnegie Corporation, 1930)                                    1
Elisha Whittelsey Fund, 1966                                                                              1
Gift of Drew Knowlton and William L. Schaeffer, in honor of Delaney H. and W. Bruce Lundberg, 2017        1
Gift of Mr. and 

## Schritt 4: Datenbereinigung

In [39]:
# Herstellung eines auf die notwendigen Spalten beschänkten Gesamtanalyse-DF
analysis_df = df[["Object ID",  "Department", "Object Name", "Title", "Culture", "Object Begin Date", "Object End Date", "Geography Type", "City", "Region", "Country", "Classification", "Medium", "Dimensions"]]
# Spalte Period fällt heraus, da sie über 80 % NaN-Wert enthält und damit für eine Analyse schwer zugänglich ist.

# Aufgrund der Eregbnisse der EDA muss keine Duplikatsbehandlung erfolgen.

In [40]:
len(analysis_df) # Zur Kontrolle: Ausgabe der length, korrekt!

448203

In [41]:
# Korrektur bzw. Vereinheitlichung der Spaltenschreibweisen
analysis_df.columns = analysis_df.columns.str.lower().str.replace(" ", "_")

In [42]:
analysis_df.head(3) # Kontrolle

,object_id,department,object_name,title,culture,object_begin_date,object_end_date,geography_type,city,region,country,classification,medium,dimensions
0,1,American Decorative Arts,Coin,One-dollar Liberty Head Coin,NaN,1853,1853,NaN,NaN,NaN,NaN,Metal,Gold,Dimensions unavailable
1,2,American Decorative Arts,Coin,Ten-dollar Liberty Head Coin,NaN,1901,1901,NaN,NaN,NaN,NaN,Metal,Gold,Dimensions unavailable
2,3,American Decorative Arts,Coin,Two-and-a-Half Dollar Coin,NaN,1909,1927,NaN,NaN,NaN,NaN,Metal,Gold,Diam. 11/16 in. (1.7 cm)


In [43]:
analysis_df.info() # Anzeige

<class 'pandas.DataFrame'>
RangeIndex: 448203 entries, 0 to 448202
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   object_id          448203 non-null  int64
 1   department         448203 non-null  str  
 2   object_name        445568 non-null  str  
 3   title              416756 non-null  str  
 4   culture            186518 non-null  str  
 5   object_begin_date  448203 non-null  int64
 6   object_end_date    448203 non-null  int64
 7   geography_type     58463 non-null   str  
 8   city               30520 non-null   str  
 9   region             31078 non-null   str  
 10  country            74450 non-null   str  
 11  classification     389924 non-null  str  
 12  medium             440155 non-null  str  
 13  dimensions         385360 non-null  str  
dtypes: int64(3), str(11)
memory usage: 112.4 MB


In [44]:
# Ersetzen der NaN-Werte
analysis_df["object_name"] = analysis_df["object_name"].fillna("unknown")
analysis_df["title"] = analysis_df["title"].fillna("unknown")
analysis_df["culture"] = analysis_df["culture"].fillna("unspecified")
analysis_df["geography_type"] = analysis_df["geography_type"].fillna("unknown")
analysis_df["city"] = analysis_df["city"].fillna("unknown")
analysis_df["region"] = analysis_df["region"].fillna("unknown")
analysis_df["country"] = analysis_df["country"].fillna("unspecified")
analysis_df["classification"] = analysis_df["classification"].fillna("unclassified")
analysis_df["medium"] = analysis_df["medium"].fillna("unknown")
analysis_df["dimensions"] = analysis_df["dimensions"].fillna("unavailable")

In [45]:
analysis_df.isna().sum() # Kontrolle: stimmt!

object_id            0
department           0
object_name          0
title                0
culture              0
object_begin_date    0
object_end_date      0
geography_type       0
city                 0
region               0
country              0
classification       0
medium               0
dimensions           0
dtype: int64

Die fehlenden Werte der analyserelevanten kategorialen Attribute wurden nicht entfernt, sondern durch aussagekräftige Platzhalterwerte ersetzt. Fehlende Angaben u. a. zu Herkunft, Kultur, Material oder Geografie wurden als "unknown" gekennzeichnet. Fehlende Klassifikationen wurden als "unclassified" ausgewiesen, fehlenden Objektdimensionen als "unavailable". Dadurch konnten alle Datensätze für die weiteren Analysen erhalten bleiben und gleichzeitig fehlende Informationen als eigene Kategorie berücksichtigt werden.

In [46]:
# Umbenennung der Jahresspalten zur besseren Übersichtlichkeit
analysis_df = analysis_df.rename(columns={"object_begin_date": "begin_year", "object_end_date": "end_year"})

In [47]:
# Bereinigung der Jahresausreißer
analysis_df[analysis_df["begin_year"] > 2026] # Anzeige

,object_id,department,object_name,title,culture,begin_year,end_year,geography_type,city,region,country,classification,medium,dimensions
349274,502176,Musical Instruments,Bell,Bell,Italian (Ancient Roman),5000,1,unknown,Rome,unknown,Italy,Idiophone-Struck-bell-without clapper,Metal,"Height 5.8 cm, Diameter 5.3 cm x 3.7 cm, Wei..."
349503,502410,Musical Instruments,Bell,Bell,Italian (Ancient Roman),5000,0,unknown,unknown,unknown,Italy,Idiophone-Struck-bell-clapper,leaded bronze,"Height 6.3 cm, Diameter +/-4.0 cm, Weight 81 g"
400509,653050,Drawings and Prints,Print,Left hand side of sheet 'The world upside dow'...,unspecified,18591861,18591861,unknown,unknown,unknown,unspecified,Prints,Woodcut,Sheet: 17 9/16 × 12 1/2 in. (44.6 × 31.8 cm)


In [48]:
# Korrektur der fehlerhaften Einträge
# Droppen der fehlerhaft datieren Objekt-IDs 502176 und 502410
analysis_df = analysis_df[(analysis_df["object_id"] != 502176) & (analysis_df["object_id"] != 502410)]

In [49]:
# Korrektur von Objekt-ID 400509
analysis_df.loc[400509, "begin_year"] = 1859
analysis_df.loc[400509, "end_year"] = 1861

Bei der Überprüfung der Datierungsvariablen wurden drei fehlerhafte bzw. unplausible Datensätze identifiziert. 
Zwei Datensätze enthielten offensichtlich inkonsistente Jahresangaben (5000, -500?), deren korrekte Werte nicht eindeutig rekonstruiert werden konnten. 
Diese Datensätze wurden daher aus der Analyse entfernt. 
Ein weiterer Datensatz enthielt die fehlerhafte Jahresangabe "18591861", die aufgrund der erkennbaren Datumsstruktur in die Werte 1859 und 1861 korrigiert wurde.

In [50]:
# Kontrolle, ob beginn_year immer vor end_year liegt bzw. mit diesem identisch ist
len(analysis_df[analysis_df["begin_year"] > analysis_df["end_year"]]) # Ergebnis 0 = korrekt!

193

In [51]:
# Kontrolle der prähistorischen Alters-/Jahresangaben
analysis_df["begin_year"].min()

np.int64(-95000)

In [52]:
analysis_df[analysis_df["begin_year"] < -2000] # Anzeige

,object_id,department,object_name,title,culture,begin_year,end_year,geography_type,city,region,country,classification,medium,dimensions
17628,22143,Arms and Armor,Ten spearheads and sinkers,Ten Spearheads and Sinkers,Japanese,-12500,-300,unknown,unknown,unknown,unspecified,Shafted Weapons,Stone,12.37.110; L. 4 3/4 in. (12.1 cm); W. 3 1/2 in...
17629,22144,Arms and Armor,Ten spearheads and sinkers,Ten Spearheads and Sinkers,Japanese,-12500,-300,unknown,unknown,unknown,unspecified,Shafted Weapons,Stone,12.37.100; L. 4 1/2 in. (11.4 cm); W. 2 1/4 in...
20999,25608,Arms and Armor,Arrowhead,Arrowhead,probably European,-10025,-4975,unknown,unknown,unknown,unspecified,Archery Equipment-Bows,"Stone, flint, bronze",unavailable
21000,25609,Arms and Armor,Implements (Celt),Implements (Celt),"Italian, Tuscany",-10000,-5000,unknown,unknown,unknown,unspecified,Miscellaneous,Stone,unavailable
21002,25611,Arms and Armor,Implements (Celt),Implements (Celt),"Italian, Tuscany",-10000,-5000,unknown,unknown,unknown,unspecified,Miscellaneous,Stone,unavailable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445300,743013,Egyptian Art,Door fragment,Fragment from a tomb door,unspecified,-2150,-1981,From,unknown,"Upper Egypt, Thebes",Egypt,unclassified,"Wood, paint",unavailable
445301,743014,Egyptian Art,Door fragment,Fragment from a tomb door,unspecified,-2150,-1981,From,unknown,"Upper Egypt, Thebes",Egypt,unclassified,"Wood, paint",unavailable
445302,743015,Egyptian Art,Door fragment,Fragment from a tomb door,unspecified,-2150,-1981,From,unknown,"Upper Egypt, Thebes",Egypt,unclassified,"Wood, paint",H. 32.3 × W. 13.7 cm (12 11/16 × 5 3/8 in.)
445303,743016,Egyptian Art,Door fragment,Fragment from a tomb door,unspecified,-2150,-1981,From,unknown,"Upper Egypt, Thebes",Egypt,unclassified,"Wood, paint",unavailable


Die Jahreswerte im Minimum- bzw. prähistorischen Bereich sind trotz ihrer zunächst ungewöhnlich erscheinenden Größe durchaus plausibel.

In [53]:
# Nochmalige Exploration der Spalte medium
analysis_df["medium"].value_counts()

medium
Commercial color lithograph                  20209
Etching                                      15038
Albumen photograph                           10389
Engraving                                     9723
Gelatin silver print                          8961
                                             ...  
Etching with red chalk                           1
Lithograph after a photograph.                   1
Lithograph after a photograph                    1
Hand-colored steel engraving                     1
Stipple engraving, etching and engraving.        1
Name: count, Length: 61445, dtype: int64

Die Spalte medium weist eine sehr hohe Anzahl unterschiedlicher Ausprägungen auf. Dies ist auf die detaillierte Erfassung von Materialien und Herstellungstechniken im Museumskontext zurückzuführen. Da die Unterschiede in vielen Fällen fachlich begründet sind, wurde auf eine umfassende Zusammenführung der Werte verzichtet.

In [54]:
# Nochmalige Exploration der Spalte classification
analysis_df["classification"].value_counts()

classification
Prints                                                69260
unclassified                                          58279
Prints|Ephemera                                       30033
Photographs                                           26821
Drawings                                              25230
                                                      ...  
Prints|Periodicals                                        1
Electrophone                                              1
Books|Ornament & Architecture|Manuscript Materials        1
Books|Portraits                                           1
Paper-Documents|Prints                                    1
Name: count, Length: 1078, dtype: int64

In [55]:
analysis_df[analysis_df["classification"].str.contains(r"\|", na=False)]["classification"].value_counts()

classification
Prints|Ephemera                                       30033
Photographs|Ephemera                                  10940
Prints|Ornament & Architecture                         4554
Drawings|Ornament & Architecture                       3910
Books|Prints|Ornament & Architecture                   3583
                                                      ...  
Drawings|Books                                            1
Prints|Periodicals                                        1
Books|Ornament & Architecture|Manuscript Materials        1
Books|Portraits                                           1
Paper-Documents|Prints                                    1
Name: count, Length: 314, dtype: int64

Das Attribut classification enthält häufig Mehrfachzuordnungen, die durch das Trennzeichen "|" gekennzeichnet sind. Da für die Analyse die übergeordnete Objektklassifikation im Vordergrund steht, wird jeweils die erste Klassifikation als Hauptkategorie übernommen. Dadurch konnten die Kategorien konsolidiert und die Vergleichbarkeit der Ergebnisse verbessert werden.

In [56]:
# Auftrennung der Klassifikation
analysis_df["classification"] = analysis_df["classification"].str.split("|").str[0]

In [57]:
analysis_df["classification"].value_counts() # Kontrolle

# Or-Werte oder /-Werte sind aufgrund ihrer gerinfgügigen Anzahl für die Analyse kein Problem.

classification
Prints             105653
unclassified        58279
Photographs         38200
Drawings            29291
Books               20441
                    ...  
Ivory                   1
games or toys           1
Bone/Ivory              1
Electrophone            1
Paper-Documents         1
Name: count, Length: 769, dtype: int64

In [58]:
analysis_df.sample(10)

,object_id,department,object_name,title,culture,begin_year,end_year,geography_type,city,region,country,classification,medium,dimensions
389214,637426,Drawings and Prints,Print,"Card No. 67, Lechwe, from the Papoose Animal G...",unspecified,1935,1935,unknown,unknown,unknown,unspecified,Prints,Commercial color lithograph,Sheet: 2 1/2 × 3 1/4 in. (6.4 × 8.2 cm)
399964,652251,Drawings and Prints,Photograph,"Louise Brooks, from the Actresses series (N245...",unspecified,1890,1890,unknown,unknown,unknown,unspecified,Photographs,Albumen photograph,Sheet: 2 1/2 × 1 7/16 in. (6.4 × 3.7 cm)
156384,254318,Greek and Roman Art,Lekythos,Lekythos,"Greek, Attic",-480,-480,unknown,unknown,unknown,unspecified,Vases,Terracotta,Overall: 9 3/16in. (23.4cm)\r\nDiameter: 3 1/2...
244802,371181,Drawings and Prints,Print,"St. Gregory of Utrecht, from the series Saints...",unspecified,1650,1650,unknown,unknown,unknown,unspecified,Prints,Engraving and etching,sheet: 17 1/8 x 11 7/8 in. (43.5 x 30.2 cm)
277415,404808,Drawings and Prints,Print,"Chinese Flee as Bombs Rake Hankow, part of the...",unspecified,1938,1938,unknown,unknown,unknown,unspecified,Prints,Commercial lithograph,sheet: 2 1/2 x 3 1/8 in. (6.3 x 8 cm)
256231,383045,Drawings and Prints,Print,Marriage Turned in Reverse,unspecified,1600,1657,unknown,unknown,unknown,unspecified,Prints,Engraving,unavailable
64003,74573,Asian Art,Piece from a kosode,unknown,Japan,1615,1868,unknown,unknown,unknown,unspecified,Textiles-Embroidered,Silk embroidery and gold leaf on silk satin,Sight: 10 7/8 x 8 11/16 in. (27.6 x 22.1 cm)
443178,740690,Islamic Art,Fragment,Fragment,unspecified,800,1300,Excavated in,Nishapur,unknown,Iran,Stucco,stucco; carved; painted,L 6.5cm x W 4.5 x H 2.5
89397,127176,Costume Institute,Cape,Cape,French,1800,1940,unknown,unknown,unknown,unspecified,unclassified,cotton,unavailable
329513,468322,Medieval Art,Tapestry,The Resurrection,South Netherlandish,1512,1528,unknown,unknown,unknown,unspecified,Textiles-Tapestries,"Wool warp; wool, silk, silver, and gilt wefts",Overall: 87 3/4 x 93 1/4in. (222.9 x 236.9cm)


In [59]:
# Technische Artefakte entfernen
analysis_df["medium"] = analysis_df["medium"].str.replace(r"\\r", "", regex=True).str.replace(r"\\n", "", regex=True)

In [60]:
analysis_df.loc[analysis_df["medium"].str.contains(r"\\r|\\n", na=False),"medium"] # Kontrolle, korrekt!

Series([], Name: medium, dtype: str)

In [61]:
# Ausgabe der statistischen Kennwerte
analysis_df.describe().round(2)

# Auffällig ist end_year.

,object_id,begin_year,end_year
count,448201.00,448201.00,448201.00
mean,346272.18,1349.32,1451.98
std,206915.19,1281.61,1018.68
min,1.00,-95000.00,-35000.00
25%,198808.00,1567.00,1600.00
50%,349815.00,1800.00,1850.00
75%,481763.00,1895.00,1910.00
max,750830.00,2016.00,157555.00


In [62]:
analysis_df[analysis_df["end_year"] > 2017]

# Die Analyse der Datierungsvariablen zeigt erhebliche Inkonsistenzen in der Spalte end_year.
# Neben plausiblen Datierungsbereichen finden sich zahlreiche offensichtlich fehlerhafte oder
# nicht nachvollziehbare Jahresangaben, die eine verlässliche Verwendung der Spalte erschweren.
# Da die historische Einordnung der Bestände bereits durch begin_year, begin_century und iran_period
# abgebildet wird, wird auf die weitere Nutzung von end_year verzichtet.

,object_id,department,object_name,title,culture,begin_year,end_year,geography_type,city,region,country,classification,medium,dimensions
42887,49818,Asian Art,Frieze section,unknown,India,1950,2099,unknown,unknown,unknown,unspecified,Sculpture,Red sandstone,H. 3 in. (7.6 cm); W. 7 15/16 in. (20.2 cm); L...
102825,188903,European Sculpture and Decorative Arts,Bowl,Wide-rimmed bowl with Vulcan Forging Arms with...,"Italian, probably Urbino",1523,15335,unknown,unknown,unknown,unspecified,Ceramics-Pottery,Maiolica (tin-glazed earthenware),Overall (confirmed): 1 1/2 × 10 1/2 in. (3.8 ×...
112025,198780,European Sculpture and Decorative Arts,Storage jar,Storage jar,"Italian, possibly Venice",1495,153505,unknown,unknown,unknown,unspecified,Ceramics-Pottery,Maiolica (tin-glazed earthenware),Overall (confirmed): 13 3/8 × 13 × 13 in. (34 ...
112028,198783,European Sculpture and Decorative Arts,Storage jar,Storage jar (one of a pair),"Italian, Venice",1545,157555,unknown,unknown,unknown,unspecified,Ceramics-Pottery,Maiolica (tin-glazed earthenware),Height: 12 5/8 in. (32.1 cm)
195591,313107,"Arts of Africa, Oceania, and the Americas",Panel,Feathered Panel,Wari,600,9900,unknown,unknown,Churunga Valley,Peru,Textiles-Featherwork,"Feathers, cotton, camelid hair",H. 30 1/2 x W. 89 x D. 16 1/4 in. (77.5 x 226...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426346,706047,The Libraries,unknown,Journal of the Royal Asiatic Society of Great ...,unspecified,1834,2099,unknown,London,unknown,England,unclassified,unknown,illustrations ; height: 8 11/16 in. (22 cm)
427784,708315,Islamic Art,Sculpture,Pregnant Haliç II,unspecified,1990,2040,Made in,Iznik,unknown,Turkey,Ceramics,Stonepaste; underglaze painted,H: 24 in (61 cm) x W: 12.5 in (31.8 cm)
429204,710689,Islamic Art,unknown,Untitled,unspecified,1964,2064,Country of Origin,unknown,unknown,United States,Drawings,Ink on paper,H: 28 1/2 in (72.4 cm) x W: 22 1/2 in (57 cm)
430298,712219,Islamic Art,Painting,After 'Khusraw and Shirin',unspecified,1959,2059,Country of Origin,unknown,unknown,Iran,Paintings,Mixed media process with archival print,H: 7.4 in (18.8 cm) x W: 2.5 in (6.4 cm)


In [63]:
# Droppen der nicht benötigen Spalte end_year
analysis_df = analysis_df.drop(columns="end_year")

Die Bereinigung ist damit (zunächst, da iterativer Prozess!) abgeschlossen.

## Schritt 4a: Feature Engineering

Zunächst wird nur zielgerichtetes Feature Engineering durchgeführt. Neben der Bereinigung und Standardisierung von Herkunftsangaben (country_clean) werden zusätzliche Analysevariablen erzeugt, darunter eine Jahrhundert-Spalte (begin_century), standardisierte Weltregionen (world_region) sowie aus der Erwerbsgeschichte (Credit Line) abgeleitete Erwerbungsinformationen und Erwerbungsjahre. Zudem werden Mehrfachangaben und historische bzw. regionale Herkunftsbezeichnungen systematisch vereinheitlicht, um geographische Analysen auf Länder-, Regional- und Kontinentebene zu ermöglichen. Für vertiefende Untersuchungen iranisch-persischer Objekte wird darüber hinaus ein eigener Teildatensatz (iran_df) angelegt. Innerhalb dieses Datensatzes wird die Variable iran_period eingeführt, um die Objekte nach zentralen historischen Epochen und Dynastien der iranischen Geschichte zu klassifizieren. Weitere Merkmale werden nur dann erzeugt, wenn sie zur Beantwortung konkreter Forschungsfragen erforderlich sind.

In [64]:
# Country Cleaning
analysis_df["country_clean"] = analysis_df["country"].fillna("unknown").astype(str).str.strip()
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace("’", "'", regex=False)

# Schutz für Namen, die zufällig "South"/"North"/"and" enthalten
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace(r"\bSouth Africa\b", "XXSOUTHAFRICAXX", regex=True)
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace(r"\bNorth America\b", "XXNORTHAMERICAXX", regex=True)
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace(r"\bWallis and Futuna Territory\b", "XXWALLISFUTUNAXX", regex=True)

# Sonderfall: Georgia, Republic of [Azerbaijan, Armenia] -> Georgia
analysis_df.loc[analysis_df["country_clean"] == "Georgia, Republic of [Azerbaijan, Armenia]", "country_clean"] = "Georgia"

# Komma-Alternativen, die sonst durch die Komma-Kürzung verlorengehen
analysis_df["country_clean"] = analysis_df["country_clean"].replace({
    "Egypt, Iran or Syria": "Egypt|Iran|Syria",
    "Iran, Iraq, or Egypt": "Iran|Iraq|Egypt",
    "Iraq, Syria, or Iran": "Iraq|Syria|Iran",
    "India, Nepal or Tibet": "India|Nepal|Tibet",
    "Iran, Syria or Turkey": "Iran|Syria|Turkey",
    "Northern Iraq, southeastern Turkey, or Syria": "Iraq|Turkey|Syria",
    "Western Iran, Northern Iraq or Northern Syria": "Iran|Iraq|Syria",
    "Syria, Yemen, or North Africa": "Syria|Yemen|North Africa",
    "Azerbaijan, Armenia or Georgia": "Azerbaijan|Armenia|Georgia",
    "Mali, Guinea, Burkina Faso, Senegal": "Mali|Guinea|Burkina Faso|Senegal",
    "Germany, France, or Bohemia": "Germany|France|Bohemia",
})

# Slash-Alternativen, die sonst durch die Slash-Kürzung verlorengehen
analysis_df["country_clean"] = analysis_df["country_clean"].replace({
    "Tibet/north India": "Tibet|India",
    "Uganda/Rwanda": "Uganda|Rwanda",
    "Uganda/Tanzania": "Uganda|Tanzania",
    "Tanzania/Kenya": "Tanzania|Kenya",
    "Tibet/Nepal?": "Tibet|Nepal",
    "France?/USA?": "France|USA",
})

# Generelle Bereinigung
analysis_df["country_clean"] = (
    analysis_df["country_clean"]
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.replace(r"\s*\(.*?\)", "", regex=True)
    .str.replace(r"\s*\(.*$", "", regex=True)
    .str.replace(r":.*$", "", regex=True)
    .str.replace(r"/.*$", "", regex=True)
    .str.replace(r",.*$", "", regex=True)
    .str.replace("?", "", regex=False)
    .str.replace(r"(?i)present-day\s+", "", regex=True)
    .str.replace(r"(?i)probably\s+", "", regex=True)
    .str.replace(r"(?i)possibly\s+", "", regex=True)
    .str.replace(
        r"\b(North|Northern|South|Southern|East|Eastern|West|Western|Central|"
        r"Northwest|Northwestern|Northeast|Northeastern|Southwest|Southwestern|"
        r"Southeast|Southeastern)\s+(?!or\b|and\b)",
        "", regex=True, case=False
    )
    .str.strip()
)

# Schutz-Platzhalter zurückholen
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace("XXSOUTHAFRICAXX", "South Africa", regex=False)
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace("XXNORTHAMERICAXX", "North America", regex=False)
analysis_df["country_clean"] = analysis_df["country_clean"].str.replace("XXWALLISFUTUNAXX", "Wallis and Futuna", regex=False)

# Orte/Regionen
analysis_df["country_clean"] = (
    analysis_df["country_clean"]
    .str.replace(r"\bBaghdad\b", "Iraq", regex=True, case=False)
    .str.replace(r"\bRajasthan\b", "India", regex=True, case=False)
    .str.replace(r"\bDeccan\b", "India", regex=True, case=False)
    .str.replace(r"\bMeaux\b", "France", regex=True, case=False)
    .str.replace(r"\bLimoges\b", "France", regex=True, case=False)
    .str.replace(r"\bCabinda\b", "Angola", regex=True, case=False)
    .str.replace(r"\bSicily\b", "Italy", regex=True, case=False)
    .str.replace(r"\bKurdistan\b", "West Asia", regex=True, case=False)
    .str.replace(r"\bJazira\b", "West Asia", regex=True, case=False)
    .str.replace(r"\bBalkans\b", "Europe", regex=True, case=False)
    .str.replace(r"\bCaucasus\b", "West Asia", regex=True, case=False)
    .str.replace(r"Republic of Cameroon", "Cameroon", regex=True, case=False)
    .str.replace(r"Sri\s?[Ll]anka", "Sri Lanka", regex=True, case=False)
)

# Regex-basierte Zuordnungen
analysis_df.loc[(analysis_df["country_clean"].str.contains("iran", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b|\|", case=False, na=False) == False), "country_clean"] = "Iran"
analysis_df.loc[(analysis_df["country_clean"].str.contains("egypt", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b|\|", case=False, na=False) == False), "country_clean"] = "Egypt"
analysis_df.loc[(analysis_df["country_clean"].str.contains("syria", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b|\|", case=False, na=False) == False), "country_clean"] = "Syria"
analysis_df.loc[(analysis_df["country_clean"].str.contains(r"turkey|anatolia", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b", case=False, na=False) == False), "country_clean"] = "Turkey"
analysis_df.loc[(analysis_df["country_clean"].str.contains("iraq", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b|\|", case=False, na=False) == False), "country_clean"] = "Iraq"
analysis_df.loc[(analysis_df["country_clean"].str.contains("afghanistan", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b|\|", case=False, na=False) == False), "country_clean"] = "Afghanistan"
analysis_df.loc[(analysis_df["country_clean"].str.contains("uzbekistan", case=False, na=False)) & (analysis_df["country_clean"].str.contains(r"\bor\b|\band\b|\|", case=False, na=False) == False), "country_clean"] = "Uzbekistan"

# Direkte Zuordnungen
analysis_df["country_clean"] = analysis_df["country_clean"].replace({
    "England": "United Kingdom", "Scotland": "United Kingdom", "UK": "United Kingdom",
    "Britian": "United Kingdom", "British Isles": "United Kingdom",
    "America": "United States", "USA": "United States", "US": "United States",
    "U.S.A.": "United States", "United States of America": "United States",
    "United State": "United States", "Unites States": "United States", "United Sates": "United States",
    "The Netherlands": "Netherlands", "the Netherlands": "Netherlands", "Holland": "Netherlands",
    "Netherland": "Netherlands", "Netherlandish": "Netherlands", "Friesland": "Netherlands",
    "Flemish": "Netherlands", "Flanders": "Netherlands",
    "Ivory Coast": "Côte d'Ivoire", "Cote d'Ivoire": "Côte d'Ivoire", "Côte d’Ivoire": "Côte d'Ivoire",
    "Republic of Benin": "Benin", "French Dahomey": "Benin", "Dahomey": "Benin",
    "Republic of Cameroon": "Cameroon", "Republic of  Cameroon": "Cameroon",
    "Republic of the Philippines": "Philippines", "Philippine Islands": "Philippines", "Philippine": "Philippines",
    "Republic of Congo": "Republic of the Congo",
    "Democratic Republic of Congo": "Democratic Republic of the Congo",
    "Dem. Rep. Congo": "Democratic Republic of the Congo",
    "Dem. Rep. of Congo": "Democratic Republic of the Congo",
    "Belgian Congo": "Democratic Republic of the Congo",
    "Congo": "Democratic Republic of the Congo",
    "French Congo": "Republic of the Congo", "Brazzaville": "Republic of the Congo",
    "Byzantine Egypt": "Egypt", "Alexandria": "Egypt", "Fustat": "Egypt",
    "Gaul": "France", "French": "France", "posssibly Gaul": "France", "Haute Garonne.": "France",
    "German": "Germany", "Alamania": "Germany", "Rhein": "Germany", "Niederbreisig": "Germany",
    "Bohemia": "Czech Republic",
    "Myanmar Burma": "Myanmar", "Burma": "Myanmar",
    "india": "India",
    "Italian": "Italy", "italy": "Italy", "Rome": "Italy",
    "Austrian": "Austria",
    "Russia Federation": "Russia", "Russian Federation": "Russia",
    "Azerbaijian": "Azerbaijan",
    "Surinam": "Suriname",
    "Republic of Madagascar": "Madagascar",
    "Herzegovina": "Bosnia", "Bosnia - Herzegovina": "Bosnia", "Bosnia and Herzegovina": "Bosnia",
    "Montenegro  Yugoslavia": "Montenegro",
    "former Yugoslavia": "Yugoslavia",
    "Macedonia": "North Macedonia",
    "Jerusalem": "Israel", "Tiberias": "Israel", "The Holy Land": "Israel", "Canaan": "Israel",
    "Caesarea": "Israel", "Latin Kingdom of Jerusalem": "Israel",
    "Early Islamic Palestine": "Palestine",
    "Damascus": "Syria", "Antioch": "Syria", "Emesa": "Syria",
    "Amman": "Jordan", "Gerasa": "Jordan",
    "excavated at al-Humayma": "Jordan", "excavated at Aqaba": "Jordan",
    "excavated ‘Ayn as-Sawda": "Jordan", "exacavated at Umm al-Walid": "Jordan",
    "excavated Dayr 'Ayn 'Abata": "Jordan", "excavated Church of Saints Peter & Paul": "Jordan",
    "excavated at the Amman Citadel": "Jordan", "excavated Kayanus Church": "Jordan",
    "excavated at al-Fudayn": "Jordan",
    "Constantinople": "Turkey", "Constaninople": "Turkey", "Byzantium": "Turkey",
    "Nicomedia": "Turkey", "Lake Van region": "Turkey",
    "Darabjird": "Iran",
    "Jazira": "West Asia", "Islamic Lands": "West Asia", "islamic lands": "West Asia",
    "Eastern Islamic lands": "West Asia", "Eastern islamic lands": "West Asia",
    "Western Islamic Lands": "West Asia", "Arabia": "West Asia", "Arabian Peninsula": "West Asia",
    "Caucasus": "West Asia", "Caucasus region": "West Asia", "Western Caucasus": "West Asia",
    "Sulawesi": "Indonesia", "Sumatra": "Indonesia", "Sarawak": "Malaysia",
    "Urundi": "Burundi", "Tunesia": "Tunisia",
    "African Republic": "Central African Republic",
    "Liberia-Côte d'Ivoire border": "West Africa",
    "Mauritania": "Mauritania",
    "Guinea Bissau": "Guinea-Bissau",
    "Republic of Timor-Leste": "Timor-Leste", "Democratic Republic of Timor-Leste": "Timor-Leste",
    "Republic of Palau": "Palau", "Federated States of Micronesia": "Micronesia",
    "Republic of Kiribati": "Kiribati", "Republic of the Marshall Islands": "Marshall Islands",
    "Kosova": "Kosovo", "Brussels": "Belgium", "Gotland": "Sweden",
    "Central Europe": "Europe", "European": "Europe",
    "Scandinavia": "Europe", "Northern Europe": "Europe",
    "Carpathian Basin region": "Europe", "Adriatic": "Europe",
    "La Tène II": "Europe", "Africa": "Africa",
    "Republic of Costa Rica": "Costa Rica", "Republic of Guinea": "Guinea",
    "Republic of Congo & Angola": "Africa", "Uganda-Sudan": "Africa",
    "Uncertain mint": "unknown", "Unknown": "unspecified", "Unknown country": "unknown",
    "Lisez et propagez nos Annales": "unknown",
    "West Asia region": "West Asia",
    "Sothern Turkmenistan": "Turkmenistan",
    "Nothern France": "France",
    "east-Europe": "Europe",
    "the Iberian peninsula": "Europe",
    "Iberian Peninsula": "Europe",
    "Lower Austria": "Austria",
    "Crete": "Greece",
    "Krinjabo": "Côte d'Ivoire",
    "Republic of Guinea-Bissau": "Guinea-Bissau",
    "New Guinea": "Papua New Guinea",
    "excavated Church of the Archangel Michael": "Jordan",
    "excavated at  Aqaba": "Jordan",
    "excavated at Umm al-Walid": "Jordan",
    "Islamic lands": "West Asia",
    "Mangareva": "French Polynesia",
    "Marquesas Islands": "French Polynesia",
    "Eastern Mediterranean": "Mediterranean",
})

# Pipe-Duplikate (gleiches Land mehrfach)
analysis_df["country_clean"] = analysis_df["country_clean"].replace({
    "Iran|Iran": "Iran", "Iran|Iran|Iran": "Iran",
    "Egypt|Egypt": "Egypt",
    "United States|United States": "United States",
    "United States|United States|United States": "United States",
    "United States|United States|United States|United States": "United States",
    "England|England": "United Kingdom",
    "Turkey|Turkey": "Turkey", "Turkey|Anatolia": "Turkey", "Anatolia|Turkey": "Turkey",
    "Iraq|Iraq": "Iraq",
    "France|France": "France",
    "Afghanistan|Afghanistan": "Afghanistan",
    "present-day Afghanistan|Afghanistan": "Afghanistan", "Afghanistan|present-day Afghanistan": "Afghanistan",
    "Georgia|Georgia": "Georgia",
    "Italy|Italy": "Italy",
    "Ecuador|Ecuador": "Ecuador",
    "Turkmenistan|Turkmenistan": "Turkmenistan",
    "Turkmenistan|Modern Turkmenistan": "Turkmenistan", "Modern Turkmenistan|Turkmenistan": "Turkmenistan",
    "Syria|Syria": "Syria",
    "Israel|Israel": "Israel"
})

# Regionale Mehrfachangaben
# Asia (übergreifend)
analysis_df.loc[analysis_df["country_clean"].str.contains(r"China\|Iran|Iran\|China|India or Iran|Iran or India|China\|India|India\|China|China\|Iraq|Iraq\|China|China or Iraq|Iraq or China|Iran or Central Asia|Central Asia or Iran|Iran\|Uzbekistan|Iran or Afghanistan|Afghanistan or Iran|Turkey\|India|India or Turkey|Turkey or Central Asia|West Asia or Central Asia|Central Asia or West Asia", case=False, na=False), "country_clean"] = "Asia"

# West Asia
analysis_df.loc[analysis_df["country_clean"].str.contains(r"Iran or Turkey|Iran or Iraq|Iraq or Syria|Syria or Iraq|Iran or Syria|Syria or Turkey|Turkey or Iraq|Iraq or Iran|Turkey or Iran|Iran\|Iraq|Iraq\|Iran|West Asia\|Iraq|Iraq\|West Asia|West Asia or Iran|Iran or West Asia|West Asia\|Iran|Iran\|West Asia|Armenia or Turkey|Turkey or Azerbaijan|Egypt\|Iran\|Syria|Iraq\|Syria\|Iran|Iran\|Syria\|Turkey|Iraq\|Turkey\|Syria|Iran\|Iraq\|Syria|Azerbaijan\|Armenia\|Georgia", case=False, na=False), "country_clean"] = "West Asia"

# Central Asia
analysis_df.loc[analysis_df["country_clean"].str.contains(r"Uzbekistan or Tajikistan|Afghanistan or Turkmenistan|Uzbekistan or Turkmenistan|Turkmenistan or Uzbekistan", case=False, na=False), "country_clean"] = "Central Asia"

# East Asia
analysis_df.loc[analysis_df["country_clean"].str.contains(r"China or Japan|Japan or China|China or Korea|Japan or Korea|China or Mongolia", case=False, na=False), "country_clean"] = "East Asia"

# South Asia
analysis_df.loc[analysis_df["country_clean"].str.contains(r"India or Pakistan|India or Nepal|Tibet\|India|Tibet\|Nepal|India\|Nepal\|Tibet|India or Sri Lanka|Sri Lanka or India", case=False, na=False), "country_clean"] = "South Asia"

# Europe
analysis_df.loc[analysis_df["country_clean"].str.contains(r"France or Germany|Germany or France|France or Spain|Spain or France|England\|France|France\|Germany|England\|Ireland|Czech Republic or Slovakia|France or Netherlands|Germany or Austria|Austria or Germany|France or England|Germany or England|England or Germany|England\|Netherlands|United Kingdom or Germany|United Kingdom or France|Ireland or United Kingdom|Norway or Sweden|Germany or Poland|France\|Italy|Germany\|Italy|France\|Switzerland|Belgium\|France|Germany or Switzerland|Netherlands or Belgium|Germany\|Netherlands|Belgium or France|England\|Germany|Germany or Netherlands|England\|Belgium|Germany\|France\|Bohemia|Italy or Germany|Germany or Italy|Italy or the Netherlands|Netherlands or Italy", case=False, na=False), "country_clean"] = "Europe"

# Africa
analysis_df.loc[analysis_df["country_clean"].str.contains(r"Côte d'Ivoire or Liberia|Gabon or Republic of Congo|Gabon or Republic of the Congo|Angola or Democratic Republic of the Congo|Democratic Republic of the Congo or Angola|Côte d'Ivoire or Ghana|Côte d'Ivoire or Burkina Faso|Mali or Niger|Liberia or Guinea|Rwanda or Burundi|Cameroon or Nigeria|Nigeria or Cameroon|Kenya or Tanzania|Senegal or Gambia|Uganda or Kenya|Ethiopia or Sudan|Nigeria or Benin|Uganda\|Rwanda|Uganda\|Tanzania|Tanzania\|Kenya|Tanzania and Kenya|Kenya and Tanzania|Mali or Ghana|Burkina Faso or Ghana|Benin or Gabon|Equatorial Guinea or Gabon|Ghana or Nigeria|Malawi or Tanzania|Sierra Leone or Mali|Nigeria or Niger|Uganda or Democratic Republic of Congo|Guinea or Mali or Burkina Faso or Côte d'Ivoire|Burkina Faso or Côte d'Ivoire or Mali|Mali or Côte d'Ivoire|Republic of Benin or Togo|Gabon or Equatorial Guinea|Togo or Ghana|Burkina Faso or Côte d'Ivoire|Republic of Cameroon or Gabon|Nigeria or Congo|Mali\|Guinea\|Burkina Faso\|Senegal|Democratic Republic of the Congo\|Angola\|Republic of the Congo|Democratic Republic of the Congo\|Republic of the Congo\|Angola|Republic of the Congo or Angola|Angola or Republic of the Congo|Mozambique\|Zimbabwe\|Malawi|Zambia or Zimbabwe|Zimbabwe or Zambia|Tanzania or Mozambique|Mozambique or Tanzania|Zimbabwe or Mozambique|Mozambique or Zimbabwe", case=False, na=False), "country_clean"] = "Africa"

# South America
analysis_df.loc[analysis_df["country_clean"].str.contains(r"Ecuador\|Peru|Peru\|Ecuador|Colombia or Ecuador|Bolivia or Peru|Peru or Ecuador|Ecuador or Peru|Colombia or Venezuela|Peru or Bolivia|Peru or Chile|Chile or Peru", case=False, na=False), "country_clean"] = "South America"

# North America
analysis_df.loc[analysis_df["country_clean"].str.contains(r"United States or Canada|Canada or United States|Costa Rica or Panama|Guatemala or Mexico|Mexico or Guatemala|United States or Mexico|Mexico\|Guatemala|Guatemala\|Mexico|France\|USA|El Salvador or Guatemala", case=False, na=False), "country_clean"] = "North America"

# North Africa
analysis_df.loc[analysis_df["country_clean"].str.contains(r"Egypt or Sudan|Egypt and Sudan", case=False, na=False), "country_clean"] = "North Africa"

# Africa / West Asia
analysis_df.loc[analysis_df["country_clean"].str.contains(r"Egypt or Palestine|Egypt or Levant|Iran\|Iraq\|Egypt|Syria\|Yemen\|North Africa", case=False, na=False), "country_clean"] = "Afrika / West Asia"

# Restliche Mehrfachangaben -> Rest-Sammelbecken
analysis_df.loc[analysis_df["country_clean"].str.contains(r"\||\bor\b|\band\b", case=False, na=False), "country_clean"] = "Transregion / Other"

analysis_df["country_clean"] = analysis_df["country_clean"].str.strip()

In [65]:
analysis_df["country_clean"].value_counts() # Kontrolle

country_clean
unspecified      373757
Egypt             32749
United States      9136
Iran               6242
Peru               3433
                  ...  
Lesotho               1
Tajikistan            1
Mauritania            1
Kosovo                1
Slovenia              1
Name: count, Length: 178, dtype: int64

In [66]:
# Ausleitung als CSV zur Kontrolle der einzelnen Werte
analysis_df["country_clean"].value_counts(dropna=False).reset_index().rename(columns={"country_clean": "country_clean","count": "count"}).to_csv("country_clean_counts.csv", index=False)

In [67]:
world_region_map = {

    # Europe
    "United Kingdom": "Europe",
    "France": "Europe",
    "Germany": "Europe",
    "Italy": "Europe",
    "Spain": "Europe",
    "Portugal": "Europe",
    "Ireland": "Europe",
    "Netherlands": "Europe",
    "Belgium": "Europe",
    "Austria": "Europe",
    "Switzerland": "Europe",
    "Czech Republic": "Europe",
    "Poland": "Europe",
    "Hungary": "Europe",
    "Croatia": "Europe",
    "Bosnia": "Europe",
    "Serbia": "Europe",
    "Slovenia": "Europe",
    "Montenegro": "Europe",
    "North Macedonia": "Europe",
    "Macedonia": "Europe",
    "Kosovo": "Europe",
    "Greece": "Europe",
    "Ukraine": "Europe",
    "Norway": "Europe",
    "Sweden": "Europe",
    "Finland": "Europe",
    "Denmark": "Europe",
    "Russia": "Europe",
    "Romania": "Europe",
    "Bulgaria": "Europe",
    "Slovakia": "Europe",
    "Albania": "Europe",
    "Malta": "Europe",
    "Cyprus": "Europe",
    "Europe": "Europe",
    "Yugoslavia": "Europe",

    # Asia
    "Iran": "Asia",
    "Iraq": "Asia",
    "Turkey": "Asia",
    "Syria": "Asia",
    "Jordan": "Asia",
    "Israel": "Asia",
    "Palestine": "Asia",
    "Lebanon": "Asia",
    "Saudi Arabia": "Asia",
    "Yemen": "Asia",
    "Armenia": "Asia",
    "Azerbaijan": "Asia",
    "Georgia": "Asia",
    "West Asia": "Asia",
    "Uzbekistan": "Asia",
    "Turkmenistan": "Asia",
    "Tajikistan": "Asia",
    "Central Asia": "Asia",
    "India": "Asia",
    "Pakistan": "Asia",
    "Afghanistan": "Asia",
    "Nepal": "Asia",
    "Bhutan": "Asia",
    "Bangladesh": "Asia",
    "Sri Lanka": "Asia",
    "Tibet": "Asia",
    "South Asia": "Asia",
    "China": "Asia",
    "Japan": "Asia",
    "Korea": "Asia",
    "Taiwan": "Asia",
    "Mongolia": "Asia",
    "East Asia": "Asia",
    "Indonesia": "Asia",
    "Malaysia": "Asia",
    "Myanmar": "Asia",
    "Thailand": "Asia",
    "Cambodia": "Asia",
    "Vietnam": "Asia",
    "Philippines": "Asia",
    "Laos": "Asia",
    "Timor-Leste": "Asia",
    "Borneo": "Asia",
    "Asia": "Asia",

    # Africa
    "Egypt": "Africa",
    "North Africa": "Africa",
    "Morocco": "Africa",
    "Algeria": "Africa",
    "Tunisia": "Africa",
    "Sudan": "Africa",
    "Ethiopia": "Africa",
    "Nigeria": "Africa",
    "Ghana": "Africa",
    "Cameroon": "Africa",
    "Benin": "Africa",
    "Togo": "Africa",
    "Mali": "Africa",
    "Senegal": "Africa",
    "Guinea": "Africa",
    "Guinea-Bissau": "Africa",
    "Sierra Leone": "Africa",
    "Liberia": "Africa",
    "Burkina Faso": "Africa",
    "Niger": "Africa",
    "Kenya": "Africa",
    "Uganda": "Africa",
    "Tanzania": "Africa",
    "Rwanda": "Africa",
    "Burundi": "Africa",
    "Angola": "Africa",
    "Gabon": "Africa",
    "Republic of the Congo": "Africa",
    "Democratic Republic of the Congo": "Africa",
    "South Africa": "Africa",
    "Zimbabwe": "Africa",
    "Zambia": "Africa",
    "Madagascar": "Africa",
    "Mauritania": "Africa",
    "Central African Republic": "Africa",
    "Côte d'Ivoire": "Africa",
    "Mozambique": "Africa",
    "Somalia": "Africa",
    "Malawi": "Africa",
    "Namibia": "Africa",
    "Chad": "Africa",
    "Equatorial Guinea": "Africa",
    "Lesotho": "Africa",
    "Gambia": "Africa",
    "Swaziland": "Africa",
    "Africa": "Africa",
    "West Africa": "Africa",

    # North America
    "United States": "North America",
    "Canada": "North America",
    "Mexico": "North America",
    "Guatemala": "North America",
    "Costa Rica": "North America",
    "Panama": "North America",
    "Honduras": "North America",
    "Nicaragua": "North America",
    "El Salvador": "North America",
    "Cuba": "North America",
    "Dominican Republic": "North America",
    "Puerto Rico": "North America",
    "Bermuda": "North America",
    "Greenland": "North America",
    "North America": "North America",

    # South America
    "Peru": "South America",
    "Colombia": "South America",
    "Ecuador": "South America",
    "Bolivia": "South America",
    "Brazil": "South America",
    "Chile": "South America",
    "Argentina": "South America",
    "Paraguay": "South America",
    "Venezuela": "South America",
    "Guyana": "South America",
    "Suriname": "South America",
    "French Guiana": "South America",

    # Oceania
    "Australia": "Oceania",
    "New Zealand": "Oceania",
    "Papua New Guinea": "Oceania",
    "New Guinea": "Oceania",
    "Fiji": "Oceania",
    "Vanuatu": "Oceania",
    "Solomon Islands": "Oceania",
    "New Caledonia": "Oceania",
    "Samoa": "Oceania",
    "Tonga": "Oceania",
    "Nauru": "Oceania",
    "Tahiti": "Oceania",
    "French Polynesia": "Oceania",
    "Kiribati": "Oceania",
    "Marshall Islands": "Oceania",
    "Palau": "Oceania",
    "Micronesia": "Oceania",
    "Cook Islands": "Oceania",
    "Marquesas Islands": "Oceania",
    "Norfolk Island": "Oceania",
    "Mangareva": "Oceania",
    "Wallis and Futuna": "Oceania",

    # Spezialfälle
    "Transregion / Other": "other",
    "Mediterranean": "other",
    "Black Sea": "other",
    "Afrika / West Asia": "other",
    "unknown": "unknwon",
    "unspecified": "unspecified"
    }

analysis_df["world_region"] = analysis_df["country_clean"].map(world_region_map).fillna("unspecified")

In [68]:
analysis_df["world_region"].value_counts()

world_region
unspecified      373846
Africa            36379
Asia              13581
North America     11673
Europe             6405
South America      4216
Oceania            1276
other               818
unknwon               7
Name: count, dtype: int64

#### Feature Engineering bzw. Bereinigung der Herkunftsangaben

Die Herkunftsangaben des Met-Datensatzes liegen in sehr unterschiedlichen Formen vor. Neben modernen Staaten finden sich historische Bezeichnungen, Regionen, Städte, Fundorte, alternative Zuschreibungen sowie uneinheitliche Schreibweisen. Für die Analyse wurden diese Angaben daher in einer bereinigten Spalte (country_clean) standardisiert und vereinheitlicht.

Länder mit unterschiedlichen Schreibweisen wurden zusammengeführt, historische oder regionale Bezeichnungen nach Möglichkeit modernen Staaten zugeordnet und Mehrfachangaben systematisch verarbeitet. Dabei wurden zusammengehörige Mehrfachangaben innerhalb derselben Weltregion nach Möglichkeit vereinheitlicht (z. B. *Iran or Turkey* → *West Asia*, *France or Germany* → *Europe*). Angaben wie "probably", "possibly" oder "(?)" wurden zu Analysezwecken dem Land zugeordnet, sofern sich daraus eine Eindeutigkeit ergab. Der Erkenntnisgewinn für die Analyse war hier der methodischen Problematik vorzuziehen.

#### Kontinentale Zuordnung

Auf Basis der bereinigten Herkunftsangaben wurden die Objekte anschließend Weltregionen zugeordnet. Dabei erfolgte die Einordnung anhand moderner geografischer Räume. Für transkontinentale Staaten mussten Einzelfallentscheidungen getroffen werden. Dies kann allerdings zu vereinzelten Problematiken führen: Als Beispiel wurden die historischen Ortsbezeichnungen *Constantinople* und *Byzantium* (das heutige Istanbul) dem heutigen Staat Türkei und in der Folge der Weltregion Asien zugewiesen.

#### Grenzen des Verfahrens

Die Bereinigung erfolgte überwiegend regelbasiert mithilfe von Mapping-Tabellen und Regex. Aufgrund der großen Vielfalt und teilweise uneinheitlichen Struktur der Herkunftsangaben konnten jedoch nicht alle Sonder- und Grenzfälle automatisiert aufgelöst werden. Verbleibende mehrdeutige oder ungewöhnliche Angaben wurden deshalb den Sammelkategorien other bzw. als unspecified als Fallback zugeordnet. Diese Kategorien enthalten überwiegend textliche und/oder strukturelle Einzelfälle und haben nur geringen Einfluss auf die allgemeinen Muster der Analyse.

Für unsere spätere Iran-Analyse kann dies in vereinzelten Fällen dazu führen, dass ein Objekt mit der beispielsweisen Landesherkunftsbezeichnung "Iran or Iraq" (-> West Asia) nicht in den Iran_DF aufgenommen wird, sofern nicht die weitere Auswahl über die Attribute culture und/oder region erfolgt.

In [69]:
# Drop der unbereinigten Country-Spalte in analysis_df
analysis_df = analysis_df.drop(columns=["country"])

In [70]:
# Auszüge aus der Credit Line, um die Erwerbsgeschichte analysegeeignet darzustellen
analysis_df["acquisition_type"] = "Other"

analysis_df.loc[df["Credit Line"].str.contains("Gift", na=False),"acquisition_type"] = "Gift"

analysis_df.loc[df["Credit Line"].str.contains("Purchase", na=False),"acquisition_type"] = "Purchase"

analysis_df.loc[df["Credit Line"].str.contains("Bequest", na=False),"acquisition_type"] = "Bequest"

analysis_df.loc[df["Credit Line"].str.contains("Transferred", na=False),"acquisition_type"] = "Transferred"

In [71]:
analysis_df["acquisition_year"] = df["Credit Line"].str.extract(r",\s*(18\d{2}|19\d{2}|20\d{2})\s*$")[0] # Sinnvoller Auszug des Jahres aus Credit Line

analysis_df["acquisition_year"] = analysis_df["acquisition_year"].fillna("unknown") # Zur Behandlung möglicher NaN-Werte

In [72]:
analysis_df["acquisition_type"].value_counts() # Kontrolle

acquisition_type
Gift           219681
Other          141081
Bequest         49942
Purchase        34970
Transferred      2527
Name: count, dtype: int64

In [73]:
analysis_df["acquisition_year"].isna().sum() # Kontrolle

np.int64(0)

In [74]:
analysis_df["acquisition_year"].value_counts() # Anzeige

acquisition_year
unknown    58316
1994       12581
1917       12007
2005        7031
1941        6997
           ...  
1878           1
1853           1
1836           1
1862           1
1863           1
Name: count, Length: 156, dtype: int64

In [75]:
sorted([year for year in analysis_df["acquisition_year"].unique() if year != "unknown" and int(year) < 1870]) # Suche nach sehr frühen jahren vor Gründung des Met

['1836', '1852', '1853', '1856', '1862', '1863', '1865']

Eine Überprüfung der Ergebnisse zeigte, dass einzelne Erwerbungsjahre bereits vor der Gründung des Metropolitan Museum of Art (1870) liegen. Da diese Werte aus den ursprünglichen Credit-Line-Angaben stammen und keine eindeutigen Datenfehler erkennen lassen, wurden sie nicht automatisch korrigiert.

In [76]:
# Engineering einer neuer Spalte für Jahrhunderte des Datierungsbeginns
def begin_century(year):
    if year > 0:
        return ((year - 1) // 100) + 1
    else:
        return -(((-year - 1) // 100) + 1)

analysis_df["begin_century"] = analysis_df["begin_year"].apply(begin_century)

# Da es historisch kein "Jahr 0" gibt, ordnet diese Funktion Objekte mit begin_year == 0 
# rechnerisch behelfsmäßig dem Wert des real nicht existierenden Jahrhunderts 0 zu.

In [77]:
analysis_df.head() # Kontrollausgabe

,object_id,department,object_name,title,culture,begin_year,geography_type,city,region,classification,medium,dimensions,country_clean,world_region,acquisition_type,acquisition_year,begin_century
0,1,American Decorative Arts,Coin,One-dollar Liberty Head Coin,unspecified,1853,unknown,unknown,unknown,Metal,Gold,Dimensions unavailable,unspecified,unspecified,Gift,1979,19
1,2,American Decorative Arts,Coin,Ten-dollar Liberty Head Coin,unspecified,1901,unknown,unknown,unknown,Metal,Gold,Dimensions unavailable,unspecified,unspecified,Gift,1980,20
2,3,American Decorative Arts,Coin,Two-and-a-Half Dollar Coin,unspecified,1909,unknown,unknown,unknown,Metal,Gold,Diam. 11/16 in. (1.7 cm),unspecified,unspecified,Gift,1967,20
3,4,American Decorative Arts,Coin,Two-and-a-Half Dollar Coin,unspecified,1909,unknown,unknown,unknown,Metal,Gold,Diam. 11/16 in. (1.7 cm),unspecified,unspecified,Gift,1967,20
4,5,American Decorative Arts,Coin,Two-and-a-Half Dollar Coin,unspecified,1909,unknown,unknown,unknown,Metal,Gold,Diam. 11/16 in. (1.7 cm),unspecified,unspecified,Gift,1967,20


In [78]:
analysis_df[analysis_df["begin_century"] < 0].head() # Kontrolle für Jahrhunderte < 0

,object_id,department,object_name,title,culture,begin_year,geography_type,city,region,classification,medium,dimensions,country_clean,world_region,acquisition_type,acquisition_year,begin_century
17582,22094,Arms and Armor,Ceremonial spearhead (Tsukushi boko),Ceremonial Spearhead (Tsukushi Boko),Japanese,-401,unknown,unknown,unknown,Shafted Weapons,Bronze,L. 33 1/2 in. (85.1 cm); W. 5 1/4 in. (13.3 cm),unspecified,unspecified,Gift,1906,-5
17628,22143,Arms and Armor,Ten spearheads and sinkers,Ten Spearheads and Sinkers,Japanese,-12500,unknown,unknown,unknown,Shafted Weapons,Stone,12.37.110; L. 4 3/4 in. (12.1 cm); W. 3 1/2 in...,unspecified,unspecified,Other,1912,-125
17629,22144,Arms and Armor,Ten spearheads and sinkers,Ten Spearheads and Sinkers,Japanese,-12500,unknown,unknown,unknown,Shafted Weapons,Stone,12.37.100; L. 4 1/2 in. (11.4 cm); W. 2 1/4 in...,unspecified,unspecified,Other,1912,-125
17630,22145,Arms and Armor,Nine celts or polished fleshers,Nine Celts or Polished Fleshers,Japanese,-1050,unknown,unknown,unknown,Archery Equipment-Bows,Stone,12.27.91; L. 7 in. (17.8 cm); W. 3/4 in. (1.9 ...,unspecified,unspecified,Other,1912,-11
20765,25370,Arms and Armor,Short sword,Short Sword,Vietnamese,-499,unknown,unknown,unknown,Swords,"Bronze, iron",L. 24 in. (61 cm),unspecified,unspecified,Other,2000,-5


In [79]:
# Herstellung eines auf die Analyse bzgl. iranischer Objekte gezielten DF
# Zunächst den DF für iranische Cultures
iran_cultures = [
    "Iran",
    "Iranian",
    "Iranian (Persian)",
    "Persian",
    "Persian, Qajar",
    "Qajar",
    "probably Iranian",
    "Indo-Persian",
    "Graeco-Persian",
    "Achaemenid",
    "Achaemenid (?)",
    "Parthian",
    "Parthian (?)",
    "Parthian or Sasanian",
    "Sasanian",
    "Sasanian (?)",
    "Sasanian or Islamic"
]

In [80]:
# Gemeinsamer iran_df (basierend auf Country, Region und Culture)
iran_df = analysis_df[(analysis_df["country_clean"] == "Iran") | (analysis_df["region"] == "Iran") | (analysis_df["culture"].isin(iran_cultures))]

In [81]:
len(iran_df) # 9093 Zeilen

9093

In [82]:
iran_df.shape # 18 Spalten: korrekt!

(9093, 17)

In [83]:
# Anzeige über .info()
iran_df.info()

# Keine NaN-Werte - Datentypen korrekt, Bestätigung der Bereinigung

<class 'pandas.DataFrame'>
Index: 9093 entries, 12834 to 444855
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   object_id         9093 non-null   int64 
 1   department        9093 non-null   str   
 2   object_name       9093 non-null   str   
 3   title             9093 non-null   str   
 4   culture           9093 non-null   str   
 5   begin_year        9093 non-null   int64 
 6   geography_type    9093 non-null   str   
 7   city              9093 non-null   str   
 8   region            9093 non-null   str   
 9   classification    9093 non-null   object
 10  medium            9093 non-null   str   
 11  dimensions        9093 non-null   str   
 12  country_clean     9093 non-null   str   
 13  world_region      9093 non-null   str   
 14  acquisition_type  9093 non-null   str   
 15  acquisition_year  9093 non-null   str   
 16  begin_century     9093 non-null   int64 
dtypes: int64(3), object(1), 

In [84]:
# Erstellung einer neuen Period-Spalte zur historischen Einordnung
def iran_period(year):
    if year < -1000:
        return "Prehistoric"
    elif year < -550:
        return "Pre-Achaemenid"
    elif year < -330:
        return "Achaemenid"
    elif year < -247:
        return "Hellenistic"
    elif year < 224:
        return "Parthian"
    elif year < 651:
        return "Sasanian"
    elif year < 1055:
        return "Early Islamic"
    elif year < 1256:
        return "Seljuk"
    elif year < 1501:
        return "Ilkhanid-Timurid"
    elif year < 1736:
        return "Safavid"
    elif year < 1796:
        return "Afsharid-Zand"
    elif year < 1925:
        return "Qajar"
    elif year < 1979:
        return "Pahlavi"
    else:
        return "Islamic Republic"


iran_df["iran_period"] = iran_df["begin_year"].apply(iran_period)

In [85]:
iran_df["iran_period"].isna().sum() # Kontrolle, korrekt!

np.int64(0)

In [86]:
iran_df["iran_period"] # Kontrolle

12834             Qajar
12836             Qajar
12837             Qajar
12838             Qajar
12839             Qajar
              ...      
444851    Early Islamic
444852    Early Islamic
444853    Early Islamic
444854    Early Islamic
444855    Early Islamic
Name: iran_period, Length: 9093, dtype: str

Zur besseren historischen Einordnung der Bestände wurde auf Basis von begin_year das zusätzliche Attribut iran_period erstellt. Als Grundlage wurde das früheste Datierungsjahr eines Objekts verwendet, da für die Zuordnung zu einer historischen Epoche eine einheitliche Datierungsvariable benötigt wird. Die Abgrenzungen orientieren sich an zentralen politischen und kulturellen Epochen bzw. Dynastien der iranischen Geschichte und reichen von der Vorgeschichte über Achämeniden, Parther, Sasaniden, Safawiden und Qajaren bis zur Pahlavi-Dynastie und der Islamischen Republik.

Test einer Datenanreicherung über die Collection-API

Zur Erweiterung des Analyse-Datensatzes wird die Collection API des Metropolitan Museum of Art genutzt. Da die im Open-Access-Datensatz enthaltenen Metadaten bereits weitgehend den verfügbaren API-Daten entsprechen, dient die API in diesem Projekt primär der Anreicherung des iranisch-persischen Analyse-Datensatzes um zusätzliche Informationen, insbesondere Bild-URLs für die spätere Visualisierung und Dashboard-Erstellung. Nach dem Test wird die Zelle auskommentiert und somit zu Dokumentationszwecken erhalten.

In [87]:
# Abruf der Bild-URLs über die Met Collection API
# iran_df["primary_image"] = ""

# for index, row in iran_df.iterrows():

    #object_id = row["object_id"]

    #url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"

    #data = requests.get(url).json()

    #iran_df.loc[index, "primary_image"] = data["primaryImage"]

## Schritt 5: Analyse

Deskriptive Vor-Analyse des Iran-Datensatzes (auch für mögliche Iteration mit EDA und Datenbereinigung)

In [88]:
iran_df[["begin_year", "begin_century"]].describe().round(2) # Statistische Werte

,begin_year,begin_century
count,9093.00,9093.00
mean,586.94,5.98
std,1027.04,10.33
min,-6000.00,-60.00
25%,200.00,2.00
50%,800.00,8.00
75%,1100.00,11.00
max,2013.00,21.00


In [89]:
iran_df[iran_df["begin_year"] == -6000] # min

,object_id,department,object_name,title,culture,begin_year,geography_type,city,region,classification,medium,dimensions,country_clean,world_region,acquisition_type,acquisition_year,begin_century,iran_period
205941,325532,Ancient Near Eastern Art,Pot,Pot,Iran,-6000,unknown,unknown,Iran,Ceramics-Vessels,Ceramic,3.07 in. (7.8 cm),unspecified,unspecified,Purchase,1962,-60,Prehistoric
205942,325533,Ancient Near Eastern Art,Cup,Cup,Iran,-6000,unknown,unknown,Iran,Ceramics-Vessels,Ceramic,3.31 in. (8.41 cm),unspecified,unspecified,Purchase,1962,-60,Prehistoric
205943,325534,Ancient Near Eastern Art,Pot,Pot,Iran,-6000,unknown,unknown,Iran,Ceramics-Vessels,Ceramic,3.19 in. (8.1 cm),unspecified,unspecified,Purchase,1962,-60,Prehistoric
205944,325535,Ancient Near Eastern Art,Pot,Pot,Iran,-6000,unknown,unknown,Iran,Ceramics-Vessels,Ceramic,3.19 in. (8.1 cm),unspecified,unspecified,Purchase,1962,-60,Prehistoric
205945,325536,Ancient Near Eastern Art,Pot,Pot,Iran,-6000,unknown,unknown,Iran,Ceramics-Vessels,Ceramic,3.07 in. (7.8 cm),unspecified,unspecified,Purchase,1962,-60,Prehistoric
205946,325537,Ancient Near Eastern Art,Pot,Pot,Iran,-6000,unknown,unknown,Iran,Ceramics-Vessels,Ceramic,3.62 in. (9.19 cm),unspecified,unspecified,Purchase,1962,-60,Prehistoric


In [90]:
iran_df[iran_df["culture"] == "Iran"].describe().round(2) # Statistische Werte von culture == Iran

,object_id,begin_year,begin_century
count,1088.00,1088.00,1088.00
mean,325154.74,-1418.48,-14.20
std,1215.70,1289.69,12.89
min,321588.00,-6000.00,-60.00
25%,324168.75,-1000.00,-10.00
50%,325230.50,-900.00,-9.00
75%,325820.75,-800.00,-8.00
max,329643.00,6.00,1.00


In [91]:
iran_df[iran_df["region"] == "Iran"].describe().round(2)

,object_id,begin_year,begin_century
count,1713.00,1713.00,1713.00
mean,324866.47,-730.68,-7.18
std,7818.60,1305.67,13.15
min,321498.00,-6000.00,-60.00
25%,323367.00,-900.00,-9.00
50%,324570.00,-600.00,-6.00
75%,325772.00,7.00,1.00
max,642369.00,1860.00,19.00


Die Analyse der Kulturangabe "Iran" zeigt, dass diese im Datensatz vorwiegend für Objekte aus vorislamischen und teilweise prähistorischen Kontexten verwendet wird. Die Bezeichnung ist daher nicht ausschließlich als Hinweis auf iranisch-persische Kulturtraditionen zu verstehen, sondern beschreibt häufig eine geografische oder archäologische Zuordnung zum Gebiet des heutigen Iran. Die Kulturangaben wurden deshalb als kuratorische Kategorien des Museums interpretiert und im weiteren Verlauf stets gemeinsam mit den Datierungsangaben betrachtet.

In [92]:
iran_df["begin_century"].value_counts().sort_index() # Kontrolle über Datierungsbeginn

begin_century
-60       6
-58       1
-53       2
-50      92
-43       2
-42       1
-40      11
-35       2
-31       8
-30      10
-29       2
-28       3
-26       3
-25       1
-24       2
-23       4
-22       1
-20      58
-19       5
-17       9
-16       9
-15      30
-14      11
-13       6
-12      14
-11       2
-10     152
-9      424
-8      203
-7       74
-6      119
-5       62
-4       28
-3       17
-2       23
-1      110
 0       32
 1      392
 2      546
 3       15
 4      143
 5       85
 6       29
 7      363
 8     2761
 9      301
 10     405
 11     362
 12     271
 13     157
 14     105
 15     225
 16     468
 17     363
 18     278
 19     266
 20      14
 21       5
Name: count, dtype: int64

Die frühesten Objekte des Datensatzes stammen aus dem Neolithikum und reichen bis etwa 6000 v. Chr. zurück. Diese Objekte machen jedoch nur einen sehr kleinen Teil der iranischen Bestände aus. Der Schwerpunkt der Sammlung liegt deutlich auf späteren historischen Perioden.

## Analyse 1: Überblick über die Gesamtsammlung (bzw. die aufgenommenen Objekte)

Ziel: Überblick über die Gesamtstruktur der Sammlung sowie Identifikation eines geeigneten Untersuchungsbereichs für die vertiefte Analyse.

In [93]:
# Anzahl unterschiedlicher Kulturen und Herkunftsländer bestimmen
print(f"In den Metadaten finden sich insgesamt {analysis_df["culture"].nunique()} Kulturangaben.")
print(f"In den Metadaten finden sich insgesamt {analysis_df["country_clean"].nunique()} Länderangaben.")

# Wie bereits die EDA gezeigt hatte, fallen unter Länder auch verschiedene andere Angaben.

In den Metadaten finden sich insgesamt 7102 Kulturangaben.
In den Metadaten finden sich insgesamt 178 Länderangaben.


In [94]:
# Häufigste Kulturangaben
analysis_df["culture"].value_counts().head(10)

culture
unspecified    261685
American        22167
French          18224
Japan           16374
China           13844
Italian          6580
Japanese         6044
British          5310
German           4798
Roman            4790
Name: count, dtype: int64

In [95]:
# DF für Diagramm ohne unspecified
top_cultures = analysis_df[analysis_df["culture"] != "unspecified"].groupby("culture")["object_id"].count().sort_values(ascending=False).head(10).reset_index()

top_cultures

,culture,object_id
0,American,22167
1,French,18224
2,Japan,16374
3,China,13844
4,Italian,6580
5,Japanese,6044
6,British,5310
7,German,4798
8,Roman,4790
9,Cypriot,2921


In [96]:
# Bar-Diagramm ohne unspecified
fig = px.bar(top_cultures, x="culture", y="object_id", text_auto=",.0f",
             title="Top 10 Listed Cultures of Origin in the Met Collection Dataset",
             labels={"culture": "Culture", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 25000])

fig.show()

# Die häufigsten dokumentierten Kulturangaben sind American, French, Japan und China. 
# Die Kategorie "unspecified" wurde in dieser Auswertung bewusst ausgeschlossen, 
# da ausschließlich hinterlegte Kulturangaben betrachtet werden.

In [97]:
# Prozentuale Verteilung an der Gesamtsammlung
culture_percent = (analysis_df["culture"].value_counts()/ len(analysis_df)* 100).round(2)

culture_percent

culture
unspecified                       58.39
American                           4.95
French                             4.07
Japan                              3.65
China                              3.09
                                  ...  
British, Staffordshire, Fenton     0.00
German, Innsbruck                  0.00
Borneo (Sarawak)                   0.00
Dayak                              0.00
French, presumably Paris           0.00
Name: count, Length: 7102, dtype: float64

In [98]:
# Bar-Diagramm mit unknwon in Prozent der Gesamtsammlung
top_cultures_percent = culture_percent.head(10).reset_index()

fig = px.bar(top_cultures_percent, x="culture", y="count", text_auto=",.2f",
             title="Top 10 Cultures of Origin in the Met Collection Dataset (%)",
             labels={"culture": "Culture", "count": "Percentage (%)"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0,100])
fig.show()

# Die Berücksichtigung der Kategorie "unspecified" relativiert die Aussagekraft von Diagramm 1:
# Selbst American, als größte dokumentierte Gruppe, weist nur einen geringen Anteil an den Gesamtobjekten auf.
# Bei über 58 % der Objekte im Datensatz ist die Herkunftskultur nicht dokumentiert.

In [99]:
# Häufigste Länderangaben
analysis_df["country_clean"].value_counts().head(10)

country_clean
unspecified       373757
Egypt              32749
United States       9136
Iran                6242
Peru                3433
France              2073
Mexico              1554
India               1529
Indonesia           1401
United Kingdom      1298
Name: count, dtype: int64

In [100]:
# DF für Diagramm ohne unspecified / unknwon
top_countries = analysis_df[(analysis_df["country_clean"] != "unspecified") & (analysis_df["country_clean"] != "unknown")].groupby("country_clean")["object_id"].count().sort_values(ascending=False).head(10).reset_index()

top_countries

,country_clean,object_id
0,Egypt,32749
1,United States,9136
2,Iran,6242
3,Peru,3433
4,France,2073
5,Mexico,1554
6,India,1529
7,Indonesia,1401
8,United Kingdom,1298
9,Germany,947


In [101]:
# Bar-Diagramm ohne unknown
fig = px.bar(top_countries, x="country_clean", y="object_id", text_auto=",.0f",
             title="Top 10 Listed Countries of Origin in the Met Collection Dataset",
             labels={"country_clean": "Country", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 35000])
fig.show()

# Ohne Berücksichtigung der Kategorie "unknown" stellen Egypt, United States und Iran
# die häufigsten dokumentierten Herkunftsländer der Sammlung dar.

In [102]:
# Prozentuale Verteilung an der Gesamtsammlung
country_percent = (analysis_df["country_clean"].value_counts()/ len(analysis_df)* 100).round(2)
country_percent

country_clean
unspecified      83.39
Egypt             7.31
United States     2.04
Iran              1.39
Peru              0.77
                 ...  
Lesotho           0.00
Tajikistan        0.00
Mauritania        0.00
Kosovo            0.00
Slovenia          0.00
Name: count, Length: 178, dtype: float64

In [103]:
# Bar-Diagramm mit unknwon in Prozent der Gesamtsammlung
top_country_percent = country_percent.head(10).reset_index()

fig = px.bar(top_country_percent, x="country_clean", y="count", text_auto=",.2f",
             title="Top 10 Countries of Origin in the Met Collection Dataset (%)",
             labels={"country_clean": "Country", "count": "Percentage (%)"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0,100])
fig.show()

# Die Berücksichtigung der Kategorie "unknown" relativiert die Aussagekraft von Diagramm 1.
# Mehr als 83 % aller Objekte besitzen keine hinterlegte Herkunftsangabe.
# Selbst Egypt als häufigstes dokumentiertes Herkunftsland macht dadurch weniger als 7 % der Gesamtbestände aus.

In [104]:
# Kontinentaler Vergleich als Bar-Diagramm
top_continent = analysis_df.groupby("world_region")["object_id"].count().sort_values(ascending=False).reset_index()

fig = px.bar(top_continent, x="world_region", y="object_id", text_auto=".,2f",
             title="World Region of Origin in the Met Collection Dataset",
             labels={"world_region": "Continent", "object_id": "Numer of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 400000])
fig.show()

In [105]:
# Kontrolle bzw. prozentuale Berechnung
(analysis_df["world_region"].value_counts()/ len(analysis_df)* 100).round(2)

world_region
unspecified      83.41
Africa            8.12
Asia              3.03
North America     2.60
Europe            1.43
South America     0.94
Oceania           0.28
other             0.18
unknwon           0.00
Name: count, dtype: float64

In [106]:
# Zeitlicher Sammlungsschwerpunkt
analysis_df["begin_year"].describe().round(2)

count    448201.00
mean       1349.32
std        1281.61
min      -95000.00
25%        1567.00
50%        1800.00
75%        1895.00
max        2016.00
Name: begin_year, dtype: float64

In [107]:
# Die Sammlung deckt einen außergewöhnlich großen Zeitraum von der Vorgeschichte bis in die Gegenwart ab.
# Deswegen wird für ein Diagramm eine Zäsur bei 1000 v. Chr. vorgenommen:
analysis_df_time = analysis_df[analysis_df["begin_year"] >= -1000]

In [108]:
# Histogramm
fig = px.histogram(analysis_df_time, x="begin_year",
                   title="Distribution of Object Dating Start (Year, from 1000 BCE – Earliest Object: 95,000 BC)",
                   labels={"begin_year": "Dating Start (Year)"})

fig.update_traces(marker_color="darkred")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Number of Objects",showgrid=False)
fig.show()

# Die Verteilung der Datierungen ist – wenig überraschend – deutlich rechtsschief.
# Das Histogramm zeigt einen klaren Sammlungsschwerpunkt auf der Neuzeit. 
# Die höchste Objektdichte liegt im 18. bis frühen 20. Jahrhundert,

In [109]:
# Rechnierische Analyse des Erwerbungsgrundes (auf Grundlage der EDA)
(analysis_df["acquisition_type"].value_counts()/ len(analysis_df)* 100).round(2)

# Schenkungen (Gift) stellen mit rund 49 % den wichtigsten Erwerbungsweg der Sammlung dar.
# Dahinter folgen die Sammelkategorie Other (31 %), Vermächtnisse (11 %) und Ankäufe (8 %), während Übertragungen nur eine geringe Rolle spielen.

acquisition_type
Gift           49.01
Other          31.48
Bequest        11.14
Purchase        7.80
Transferred     0.56
Name: count, dtype: float64

### Ergebnis Analyse 1:

Die Sammlung des Metropolitan Museum of Art zeichnet sich durch eine hohe kulturelle, geografische und zeitliche Vielfalt aus. Gleichzeitig zeigen die Metadaten deutliche Dokumentationslücken: Für rund 58 % der Objekte ist keine Kulturangabe sowie für etwa 83 % kein Herkunftsland und – daran anschließend – kein Herkunftskontinent dokumentiert.

Innerhalb der dokumentierten Kulturangaben sind am meisten amerikanische (4,95 %), französische (4,07 %), japanische (3,65 %) und chinesische (3,09 %) Objekte vertreten. Bei den Herkunftsländern entfallen 7,31 % der Gesamtbestände auf Ägypten, 2,04 % auf die Vereinigten Staaten und 1,39 % auf Iran. Auf kontinentaler Ebene weist Afrika mit 8,12 % den größten Anteil dokumentierter Herkunftsangaben auf, gefolgt von Asien (3,03 %) und Nordamerika (2,60 %).

Diese Werte verdeutlichen bestimmte Sammlungsschwerpunkte innerhalb des dokumentierten Teilbestands, erlauben aufgrund der hohen Anteile fehlender Angaben jedoch nur sehr bedingt Rückschlüsse auf die Gesamtverteilung aller Objekte.

Zeitlich umfasst die Sammlung einen außergewöhnlich langen Zeitraum von der Vor- und Frühgeschichte bis in die Gegenwart. Die Objektdichte konzentriert sich jedoch deutlich auf die Neuzeit, insbesondere auf das 18. bis frühe 20. Jahrhundert. Fast die Hälte aller Objekte ergignen als Schenkungen an das Museum.

Aufgrund der Größe und Heterogenität der Gesamtbestände erfolgt die vertiefte Untersuchung im weiteren Verlauf anhand der iranisch-persischen Bestände.

## Analyse 2: Der iranisch-persische Kultur- bzw. Herkunftsraum im Kontext der Gesamtsammlung

Ziel: Einordnung der iranisch-persischen Bestände innerhalb der Gesamtstruktur des Museums sowie transparente Herleitung des Untersuchungsdatensatzes.

In [110]:
# Anteil der iranisch-persischen Bestände an der Gesamtsammlung bestimmen
iran_percent = round(len(iran_df) / len(analysis_df) * 100, 2)
print(f"Der Anteil der definierten iranisch-persischen Bestände am Datensatz beträgt {iran_percent} %.")

Der Anteil der definierten iranisch-persischen Bestände am Datensatz beträgt 2.03 %.


In [111]:
# Objekte im Datensatz nach dem Jahr -1000 (wissenschaftliche Annäherung: Einwanderung indo-europäischer Gruppen in das iranische Hochland)
iran_pre = iran_df[iran_df["begin_year"] >= -1000]
print(len(iran_pre))

8798


In [112]:
# Darstellung des zahlenmßigen Anteils der Objekte des Iran_DF am Gesamtdatensatz je Zeitabschnitt auf Basis des entsprechenden Diagramms in Analyse 1
analysis_df_time["dataset"] = "Entire Collection"

iran_pre["dataset"] = "Iran_DF"

overlay_df = pd.concat([analysis_df_time[["begin_year", "dataset"]], iran_pre[["begin_year", "dataset"]]])

fig = px.histogram(overlay_df, x="begin_year", color="dataset", barmode="overlay", opacity=0.7,
                   title="Distribution of Object Dating Start (Year, from 1000 BCE): Entire Collection vs. Iran Dataset",
                   labels={"begin_year": "Dating Start (Year)", "dataset": "Dataset"})

fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Number of Objects", showgrid=False)
fig.show()

# Das Overlay bestätigt zunächst, dass die iranisch-persischen Bestände einen kleineren Teil der Gesamtbestände ausmachen.
# Gleichzeitig fällt ein außergewöhnlich hoher Anteil iranisch-persischer Objekte im Zeitraum 800–809 n. Chr. auf, 
# der in die iran_period Early Islamic fällt.

In [113]:
# Anteil iran_pre an iran_df
round(len(iran_pre) / len(iran_df) * 100, 2)
# 96,76 % des ausgewählten iran_df stammen aus der engeren Zeit der iranisch-persischen Geschichte.

96.76

In [114]:
# Anteil der parameter Country, Region und Culture am Iran_DF
country_df = analysis_df[analysis_df["country_clean"] == "Iran"]
region_df = analysis_df[analysis_df["region"] == "Iran"]
culture_df = analysis_df[analysis_df["culture"].isin(iran_cultures)]

country_count = len(country_df)
region_count = len(region_df)
culture_count = len(culture_df)

print(f"Country: {country_count}")
print(f"Region: {region_count}")
print(f"Culture: {culture_count}")
print(f"Vergleich der drei Counts addiert: {country_count + region_count + culture_count}.")
print(f"Der Iran-DF enthält tatsächlich {len(iran_df)} Objekte.") # Abweichende Zahl: Mehrfachzuordnung über verschiedene Auswahlkriterien.

Country: 6242
Region: 1713
Culture: 2742
Vergleich der drei Counts addiert: 10697.
Der Iran-DF enthält tatsächlich 9093 Objekte.


In [115]:
# Bar-Diagramm
fig = px.bar(x=["Country", "Culture", "Region"], y=[country_count, culture_count, region_count], text_auto=",.0f",
             title="Contribution of Selection Criteria to the Iran Dataset",
             labels={"x": "Selection Criterion", "y": "Number of Objects"})

fig.update_traces(marker_color="darkred",textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 7000])
fig.show()

# Die Konstruktion des Iran-Datensatzes beruht auf drei unterschiedlichen Selektionskriterien.
# Da einzelne Objekte gleichzeitig über Country, Region und/oder Culture dem iranisch-persischen Kulturraum zugeordnet werden können, 
# entstehen Mehrfachzuordnungen. Die Summe der drei Kategorien übersteigt daher die tatsächliche Größe des Iran-Datensatzes von 9093 Objekten.
# Den größten Beitrag zur Konstruktion des Datensatzes leistet das Kriterium Country, gefolgt von Culture und Region.

In [116]:
# Auswertung der Länderwerte
iran_df["country_clean"].value_counts()

# Die zusätzlichen Objekte, die über Culture- und Region-Kriterien in den Iran-Datensatz aufgenommen werden, 
# führen nicht zu weiteren eindeutig identifizierbaren Herkunftsländern.

country_clean
Iran           6242
unspecified    2851
Name: count, dtype: int64

In [117]:
# Aswertung der Kontinentalwerte
iran_df["world_region"].value_counts()

# Diese folgen den Werten der Herkunftsländer.

world_region
Asia           6242
unspecified    2851
Name: count, dtype: int64

In [118]:
# Ausgabe der Regionswerte
iran_df["region"].value_counts()

# Die zusätzlichen Regionsangaben (sofern vorhanden) zeigen eine Verbreirtung der Objekte über den gesamten Raum an,
# der historisch stark von der iranisch-persischen Kultur geprägt wurde.

# Interessant, aber unproblematisch, sind einzelne Ausreißer (beispielsweise Poland).

region
unknown                              6467
Iran                                 1713
Mesopotamia                           339
Iran or Mesopotamia                   187
Iran, probably from Luristan          113
Iran, Luristan                         59
Northwestern Iran                      42
Western Iran                           36
Iran                                   35
Northwestern Iran, Caspian region      21
Iran (?)                               17
Khurasan                               15
Western Asia                            6
Northern Iran                           5
Mesopotamia or Iran                     4
Southwestern Iran                       4
Central Asia or Iran                    3
Egypt                                   2
Transcaucasia or Iran                   2
Anatolia or Iran                        2
Central Asia                            2
probably Khurasan                       2
Iran or Northern Mesopotamia            1
Eurasian steppes (?)       

In [119]:
# Beispielhafte Anzeige region == Poland
iran_df[iran_df["region"] == "Poland"]

# Es handelt sich um ein Rhyton aus sasanidischer Zeit.
# Die Regionsangabe "Poland" steht offensichtlich in keinem Zusammenhang mit der historischen Entstehung des Objekts.
# Auf Grundlage der vorliegenden Metadaten kann die Ursache dieser Zuordnung nicht näher bestimmt werden.

,object_id,department,object_name,title,culture,begin_year,geography_type,city,region,classification,medium,dimensions,country_clean,world_region,acquisition_type,acquisition_year,begin_century,iran_period
204525,324024,Ancient Near Eastern Art,Rhyton,Rhyton in the form of a Saiga antelope head,Sasanian,400,unknown,unknown,Poland,Metalwork-Vessels,Silver-gilt,11.1 x 7.09 in. (28.19 x 18.01 cm),unspecified,unspecified,Other,1947,4,Sasanian


In [120]:
# Analyse der City im Iran_DF
iran_df["city"].value_counts()

city
unknown                               4557
Nishapur                              3676
Nishapur|Nishapur                      131
Kashan                                 110
Rayy                                    95
                                      ... 
Jajarm                                   1
Isfahan|Rayy                             1
Nihavand                                 1
possibly Rayy                            1
possibly from the vicinity of Rayy       1
Name: count, Length: 81, dtype: int64

In [121]:
# Ausleitung als CSV
iran_df["city"].value_counts(dropna=False).reset_index().rename(columns={"city": "city","count": "count"}).to_csv("city_counts.csv", index=False)

Das Attribut city umfasst zahlreiche Detailangaben, darunter offensichtliche Dubletten (z. B. "Nishapur|Nishapur") sowie unsichere Ortszuweisungen mit Zusätzen wie "probably" oder "possibly". Da diese Angaben in vielen Fällen eindeutig auf dieselbe Stadt verweisen, werden sie für die Analyse vereinheitlicht.

Auf eine Zusammenführung von Mehrfachzuordnungen wie "Isfahan or Kashan" oder "Qazvin or Mashhad" wird dagegen verzichtet, da hier keine eindeutige Zuordnung zu einem einzelnen Ort möglich ist. Ziel der Bereinigung ist daher nicht die vollständige Harmonisierung aller Ortsangaben, sondern die Zusammenführung offensichtlicher Schreib- und Unsicherheitsvarianten, um die wichtigsten historischen Zentren des iranisch-persischen Kulturraums konsistenter darzustellen. Korrekturen der Schreibweisen erfolgen sonst nur im Zuge weiterer Bereinigungsschritte.

In [122]:
# Vereinheitlichung – iterative Datenbereinigung
iran_df["city"] = iran_df["city"].replace({

    # Nishapur
    "Nishapur|Nishapur": "Nishapur",
    "probably Nishapur": "Nishapur",
    "possibly Nishapur": "Nishapur",
    "Jovain|Nishapur": "Nishapur",
    "Nishapur-Falaki": "Nishapur",

    # Kashan
    "probably Kashan": "Kashan",

    # Isfahan
    "probably Isfahan": "Isfahan",

    # Rayy
    "probably Rayy": "Rayy",
    "possibly Rayy": "Rayy",
    "possibly from the vicinity of Rayy": "Rayy",

    # Gurgan
    "probably Gurgan": "Gurgan",
    "possibly Gurgan": "Gurgan",

    # Tabriz
    "probably Tabriz": "Tabriz",
    "possibly Tabriz": "Tabriz",

    # Shiraz
    "probably Shiraz": "Shiraz",
    "possibly Shiraz": "Shiraz",

    # Kirman
    "probably Kirman": "Kerman",

    # Hamadan
    "probably Hamadan": "Hamadan",

    # Mashhad
    "probably Mashhad": "Mashhad",
    "possibly Mashhad": "Mashhad",

    # Amul
    "possibly Amul": "Amul",

    # Qazvin
    "probably Qazvin": "Qazvin",
    "possibly Qazvin": "Qazvin",

    # Sultanabad
    "probably Sultanabad": "Sultanabad",

    # Feraghan
    "probably Feraghan": "Feraghan",

    # Qasr-e Abu Nasr
    "probably Takht-i Sulaiman": "Qasr-e Abu Nasr",
    "Qasr-i Abu Nasr": "Qasr-e Abu Nasr",

    # Sonderfälle
    "Isfahan|Rayy": "Isfahan or Rayy",
    "Amul, Hamadan or Zenjan": "Amul, Hamadan or Zenjan",
    "Nishapur|al-Muhammadiyya": "Nishapur or Rayy",
    "probably Kashan, possibly Isfahan": "Kashan or Isfahan",
    "Kirmanshah": "Kermanshah",
    "Kirman": "Kerman"
})

iran_df["city"] = (iran_df["city"].str.replace("probably ", "", regex=False).str.replace("possibly ", "", regex=False)) # Alle Sonderfälle ausschließen

# Doppelnennung vereinheitlichen
iran_df["city"] = iran_df["city"].replace({

    "Isfahan or Kashan": "Isfahan or Kashan",
    "Kashan or Isfahan": "Isfahan or Kashan",

    "Isfahan or Kirman": "Isfahan or Kerman",
    "Kirman or Isfahan": "Isfahan or Kerman",

    "Isfahan or Rayy": "Isfahan or Rayy",
    "Rayy or Isfahan": "Isfahan or Rayy",

    "Qazvin or Mashhad": "Mashhad or Qazvin",
    "Mashhad or Qazvin": "Mashhad or Qazvin"
})

In [123]:
# Nochmalige Ausgabe der Werte in city nach Bereinigung
iran_df["city"].value_counts()

# Sofern konkrete Ortsangaben vorliegen, konzentrieren sich die iranisch-persischen Bestände 
# stark auf wenige historische Zentren des iranischen Kulturraums.
# Besonders dominant ist Nishapur mit 3.822 Objekten, gefolgt von Kashan, Isfahan, Rayy und Tabriz.
# Gleichzeitig besitzt rund die Hälfte der Objekte keine dokumentierte Ortsangabe.

city
unknown                    4557
Nishapur                   3822
Kashan                      162
Isfahan                     109
Rayy                        103
Tabriz                       98
Shiraz                       56
Qasr-e Abu Nasr              39
Gurgan                       31
Kerman                       17
Qazvin                       13
Hamadan                       9
Mashhad                       8
Isfahan or Kashan             6
Sultanabad                    6
Amul                          5
Tehran                        4
Varamin                       4
Isfahan or Kerman             4
Yazd                          4
Kermanshah                    3
Bakshis                       2
Saraband                      2
Herat                         2
Natanz                        2
Feraghan                      2
Nishapur or Rayy              2
Shirvan                       1
Feragham                      1
Jushagan                      1
Bali                          1
Ras

In [124]:
# Als Bar-Diagramm (Top 10)
city_top_10 = iran_df["city"].value_counts().head(10).reset_index()

fig = px.bar(city_top_10, x="city",y="count", text_auto=",.0f",
             title="Top 10 Cities of Origin (Iranian-Persian Objects)",
             labels={"city": "City", "count": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 5000])
fig.show()

# Optische Bestätigung der tabellarischen Rechnung.
# Bei den bekannten Städteangaben dominiert Nishapur.
# Dennoch bleibt zu bedenken, dass für die Hälfte aller Objekte des Iran_DF 
# kein Stadtbezug im Datensatz dokumentiert ist. 

### Ergebnis Analyse 2

Der definierte Iran-Datensatz umfasst 9.093 Objekte und macht damit rund 2 % der Gesamtbestände des Metropolitan Museum of Art bzw. des zugrunde liegenden Datensatzes aus. Besonders im Zeitraum um das frühe 9. Jahrhundert n. Chr. treten iranisch-persische Objekte überproportional häufig auf und weisen damit bereits vor der Detailanalyse auf einen spezifischen historischen Sammlungsschwerpunkt hin.

Die Konstruktion des Datensatzes basiert auf einer Kombination geografischer (country_clean, region) und kulturhistorischer (culture) Selektionskriterien. Die dokumentierten Ortsangaben konzentrieren sich zudem stark auf wenige historische Zentren des iranisch-persischen Kulturraums, insbesondere auf Nishapur, Kashan, Isfahan, Rayy und Tabriz.

Da einzelne Objekte mehreren Kriterien gleichzeitig zugeordnet werden können, entstehen Mehrfachzuordnungen, sodass die Summe der Einzelkategorien die tatsächliche Größe des Iran-DF übersteigt. Gleichzeitig ist zu berücksichtigen, dass aufgrund von Datenlücken und Inkonsistenzen in verschiedenen Attributen keine vollständige Sicherheit darüber besteht, dass sämtliche Iran-bezogenen Objekte erfasst werden konnten.

Für die historische Einordnung wird ergänzend eine Zäsur um 1000 v. Chr. (als analytische Arbeitsdefinition) berücksichtigt, da die wissenschaftliche Forschung die Ansiedlung indo-europäischer bzw. iranischer Sprachgruppen auf dem iranischen Hochland überwiegend in den Übergang vom späten 2. zum frühen 1. Jahrtausend v. Chr. datiert. Ältere Objekte bleiben dennoch Bestandteil der Analyse, da sie sowohl aus Sicht der Museumsdokumentation als auch der Kulturgeschichte Irans relevant sind. Insgesamt stammen 8.798 Objekte (96,76 % des Iran-DF) aus der Zeit nach 1000 v. Chr. und damit aus dem engeren Zeitraum der iranisch-persischen Geschichte.

Der untersuchte Datensatz stellt somit eine ausreichend große und zugleich fachlich nachvollziehbar abgegrenzte Grundlage für die nachfolgenden Analysen dar.

## Analyse 3: Zeitliche Schwerpunkte und historische Epochen

Ziel: Identifikation historischer Sammlungsschwerpunkte.

In [125]:
# Früheste und späteste Datierung
print(f"Früheste Datierung: {iran_df["begin_year"].min()}.")
print(f"Späteste Datierung: {iran_df["begin_year"].max()}.")

# Die Datierungen der iranisch-persischen Bestände reichen von 6000 v. Chr. bis 2013 n. Chr. 
# und decken damit einen außergewöhnlich langen historischen Zeitraum ab.

Früheste Datierung: -6000.
Späteste Datierung: 2013.


In [126]:
# Durchschnitt und Median
print(f"Durchschnittliche Datierung: {round(iran_df["begin_year"].mean(), 2)}.")
print(f"Median der Datierung: {iran_df["begin_year"].median()}.")

# Der Median von 800 n. Chr. zeigt jedoch, dass sich die Bestände deutlich auf spätere historische Epochen konzentrieren.

Durchschnittliche Datierung: 586.94.
Median der Datierung: 800.0.


In [127]:
# Verteilung nach iran_period
iran_df["iran_period"].value_counts()

# Mit 3836 Objekten stellt die Early Islamic Period die mit Abstand größte Objektgruppe dar 
# und prägt die zeitliche Struktur der Sammlung wesentlich stärker als alle anderen Epochen.

# Dies deckt sich mit dem Overaly in Analyse 2.

iran_period
Early Islamic       3836
Parthian            1105
Pre-Achaemenid       930
Safavid              901
Seljuk               660
Ilkhanid-Timurid     451
Qajar                433
Prehistoric          295
Sasanian             274
Achaemenid           126
Afsharid-Zand         43
Hellenistic           22
Pahlavi               10
Islamic Republic       7
Name: count, dtype: int64

In [128]:
# Bar-Digramm 
century_df = iran_df.groupby("begin_century")["object_id"].count().reset_index()

fig = px.bar(century_df, x="begin_century", y="object_id", text_auto=",.0f",
             title="Iranian-Persian Objects by Dating Start Century",
             labels={"begin_century": "Dating Start Century", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 3000])
fig.show()

# Das Diagramm zeigt die große zeitliche Variabilität des Iran_DF, 
# ist allerdings unübersichtlich.

In [129]:
# Bar-Digramm ab 1000 v. Chr.
century_without_pre_df = iran_df[iran_df["iran_period"] != "Prehistoric"].groupby("begin_century")["object_id"].count().reset_index()

fig = px.bar(century_without_pre_df, x="begin_century", y="object_id", text_auto=",.0f",
             title="Iranian-Persian Objects by Dating Start Century (without Prehistoric Period)",
             labels={"begin_century": "Dating Start Century", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 3000])
fig.show()

# Sehr gut zu sehen ist, dass die meisten Objekte des Iran_DF aus dem 8. Jahrhundert stammen,

In [130]:
century_df["percentage"] = (century_df["object_id"]/ century_df["object_id"].sum()* 100).round(2)

century_df.sort_values("percentage", ascending=False).head(10)

# Bestätigt durch Rechnung: Das 8. Jahrhundert dominiert deutlich mit über 30 % des gesamten Iran_DF.
# Auffällig ist zudem er relativ große Anteil an Objekten aus dem 9. Jahrhundert v. Chr.

,begin_century,object_id,percentage
44,8,2761,30.36
38,2,546,6.00
52,16,468,5.15
27,-9,424,4.66
46,10,405,4.45
37,1,392,4.31
53,17,363,3.99
43,7,363,3.99
47,11,362,3.98
45,9,301,3.31


In [131]:
# Vorbereitung für ein Liniendiagramm nach iran_period in korrekter historischer Reihenfolge
iran_period_line_df = iran_df.groupby("iran_period").agg(object_id=("object_id", "count"), min_year=("begin_year", "min"), max_year=("begin_year", "max")).reset_index()

iran_period_line_df = iran_period_line_df.set_index("iran_period")

iran_period_line_df = iran_period_line_df.loc[
    [
        "Prehistoric",
        "Pre-Achaemenid",
        "Achaemenid",
        "Hellenistic",
        "Parthian",
        "Sasanian",
        "Early Islamic",
        "Seljuk",
        "Ilkhanid-Timurid",
        "Safavid",
        "Afsharid-Zand",
        "Qajar",
        "Pahlavi",
        "Islamic Republic"]]

iran_period_line_df = iran_period_line_df.reset_index()

iran_period_line_df["percentage"] = iran_period_line_df["percentage"] = (iran_period_line_df["object_id"] / iran_period_line_df["object_id"].sum() * 100).round(2)

In [132]:
# Liniendiagramm
fig = px.line(iran_period_line_df, x="iran_period", y="object_id", markers=True, hover_data=["percentage", "min_year", "max_year"],
              title="Chronological Distribution of Iranian-Persian Objects",
              labels={"iran_period": "Historical Period", "object_id": "Number of Objects"})

fig.update_traces(line_color="darkred")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False,tickangle=-45)
fig.update_yaxes(showgrid=False)
fig.show()

# Die Verteilung der Objekte über die iranischen Epochen verläuft deutlich ungleichmäßig. 
# Die Early Islamic Period stellt den mit Abstand größten Sammlungsschwerpunkt dar, 
# während zahlreiche andere Epochen nur einen vergleichsweise kleinen Anteil der Bestände ausmachen.

In [133]:
# Bar-Diagramm nach historischer Verteilung, absteigend sortiert
iran_period_df = iran_df.groupby("iran_period")["object_id"].count().sort_values(ascending=False).reset_index()

fig = px.bar(iran_period_df, x="iran_period", y="object_id", text_auto=",.0f",
             title="Iranian-Persian Objects by Historical Period",
             labels={"iran_period": "Historical Period", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False, range=[0,5000])
fig.show()

# Bestätigung: Die Bestände konzentrieren sich stark auf wenige historische Kernperioden.
# Mit 3.836 Objekten stammt der größte Teil der iranisch-persischen Bestände aus der Early Islamic Period, 
# gefolgt von den Parthern (1.105) und den vorachämenidischen Beständen (930).

In [134]:
iran_period_percent = (iran_df["iran_period"].value_counts() / len(iran_df) * 100).round(2).sort_values(ascending=False).reset_index()

fig = px.bar(iran_period_percent, x="iran_period", y="count", text_auto=".2f",
             title="Iranian-Persian Objects by Historical Period (%)",
             labels={"iran_period": "Historical Period", "count": "Percentage (%)"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5,template="plotly_white")
fig.update_xaxes(showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False,range=[0, 50])
fig.show()

# Die Early Islamic Period macht mit 42,19 % mehr als zwei Fünftel des gesamten Iran-Datensatzes aus.
# Zusammen entfallen über 60 % der Bestände auf die Early Islamic, Parthian und Pre-Achaemenid Periods.

In [135]:
# Untersuchung der Datierung innerhalb der Early Islamic Period
early_islamic_df = iran_df[iran_df["iran_period"] == "Early Islamic"]

fig = px.histogram(early_islamic_df, x="begin_year",
                   title="Distribution of Object Dating Start (Year) within the Early Islamic Period",
                   labels={"begin_year": "Dating Start (Year)"})

fig.update_traces(marker_color="darkred")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Number of Objects", showgrid=False)
fig.show()

# Die Dominanz der Early Islamic Period verteilt sich nicht gleichmäßig über die gesamte Epoche.
# Stattdessen konzentrieren sich die Bestände außergewöhnlich stark auf den Zeitraum 800–809 n. Chr., in den allein rund 2.525 Objekte fallen.
# Die Early Islamic Period wird somit nicht nur durch ihre hohe Gesamtzahl an Objekten geprägt, sondern zusätzlich durch einen ausgeprägten Sammlungsschwerpunkt im frühen 9. Jahrhundert.

### Ergebnis Analyse 3

Die iranisch-persischen Bestände decken einen Zeitraum von 6000 v. Chr. bis 2013 n. Chr. ab und umfassen damit nahezu die gesamte Kulturgeschichte des iranischen Raums. Die zeitliche Verteilung der Objekte ist jedoch deutlich ungleichmäßig. Sowohl die Analyse nach Jahrhunderten als auch die Auswertung der historischen Epochen zeigen eine starke Konzentration auf wenige Kernperioden. Mit 3.836 Objekten bzw. 42,19 % aller Bestände stellt die Early Islamic Period den mit Abstand wichtigsten Sammlungsschwerpunkt dar. Daneben sind insbesondere die Parther-, vorachämenidischen und safawidischen Perioden stark vertreten, während andere Epochen nur einen vergleichsweise kleinen Anteil der Bestände ausmachen.

Innerhalb der Early Islamic Period konzentrieren sich die Bestände zudem außergewöhnlich stark auf den Zeitraum 800–809 n. Chr. Allein auf dieses Jahrzehnt entfallen 2525 Objekte, wodurch sich der bereits in Analyse 2 identifizierte Sammlungsschwerpunkt zusätzlich bestätigt.

Die iranisch-persischen Bestände des Metropolitan Museum of Art bilden somit die Geschichte des iranischen Kulturraums über einen sehr langen Zeitraum ab, konzentrieren sich jedoch deutlich auf wenige historische Schwerpunkte.

Auffällig ist, dass unter den drei iranischen Großreichen der Antike die Parther mit 1.105 Objekten deutlich stärker vertreten sind als die Achämeniden (126) und die Sasaniden (274). Dies erscheint bemerkenswert, da letztere in der heutigen öffentlichen Wahrnehmung sowie in historischen Darstellungen meist deutlich präsenter sind als die Parther. Die Museumsbestände verleihen der parthischen Periode einen eigenen sammlungs- und objektbezogenen Schwerpunkt innerhalb der iranischen Geschichte.

## Analyse 4: Materielle und funktionale Zusammensetzung

Ziel: Analyse der materiellen und funktionalen Zusammensetzung der iranisch-persischen Bestände.

In [136]:
# Häufigste Objektarten der iranisch-persischen bestände
iran_df["object_name"].value_counts()

# Insgesamt 679 verschiedene Objektarten (durch unterschiedliche Datenaufnahmen teilweise Doppelungen / Einzelfälle)

object_name
Fragment                                2126
Coin                                     715
Bowl                                     616
Stamp seal                               325
Folio from an illustrated manuscript     236
                                        ... 
Saddle Bag                                 1
Illustrated manuscript, folio              1
Album leaf, illustrated                    1
Goblets                                    1
Double album leaf, non-illustrated         1
Name: count, Length: 679, dtype: int64

In [137]:
iran_df["object_name"].value_counts().head(10)

object_name
Fragment                                2126
Coin                                     715
Bowl                                     616
Stamp seal                               325
Folio from an illustrated manuscript     236
Jar                                      214
Sealing                                  198
Ewer                                     182
Bottle                                   165
Dish                                     143
Name: count, dtype: int64

Die Spalte object_name umfasst insgesamt 679 verschiedene Objektarten. Eine vollständige Harmonisierung aller Ausprägungen würde einen erheblichen manuellen und regelbasierten Bereinigungsaufwand erfordern, dessen Nutzen für die vorliegende Analyse begrenzt wäre. Die Sichtung der häufigsten Objektarten zeigt keine auffälligen Dubletten oder inkonsistenten Schreibweisen innerhalb der dominierenden Objektgruppen. Da die zentralen Sammlungsschwerpunkte bereits anhand der häufigsten Objektarten eindeutig erkennbar sind, wurde auf eine weitergehende Standardisierung verzichtet. Mögliche Abweichungen in selten auftretenden Objektbezeichnungen haben daher keinen wesentlichen Einfluss auf die grundlegenden Aussagen der Analyse.

In [138]:
# Bar-Diagramm der Top 10 Objektarten
top_objects = iran_df.groupby("object_name")["object_id"].count().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(top_objects, x="object_name", y="object_id", text_auto=",.0f",
             title="Top 10 Object Types in the Iranian-Persian Dataset",
             labels={"object_name": "Object Type", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False, range=[0, 2500])
fig.show()

# Die iranisch-persischen Bestände werden deutlich von Fragmenten dominiert, die allein über 2100 Objekte umfassen.
# Daneben zählen Münzen, Schalen und Siegel zu den häufigsten Objektarten der Sammlung.

In [139]:
# Top 10 der Objektklassifikationen
iran_df["classification"].value_counts().head(10)

classification
Stucco                 1745
Ceramics               1463
Coins                   652
Codices                 481
Ceramics-Vessels        468
Metalwork-Ornaments     338
Glass                   280
Stone-Stamp Seals       241
Metal                   232
Ivories and Bone        215
Name: count, dtype: int64

In [140]:
top_classifications = iran_df.groupby("classification")["object_id"].count().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(top_classifications, x="classification", y="object_id", text_auto=",.0f",
             title="Top 10 Classifications in the Iranian-Persian Dataset",
             labels={"classification": "Classification", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False, range=[0, 2000])
fig.show()

# Die Analyse der Klassifikationen ist aufgrund der vorangegangenen Bereinigung gut interpretierbar.
# Die Bestände bestehen überwiegend aus Stuckobjekten und Keramik, die gemeinsam bereits mehr als 3200 Objekte umfassen.
# Münzen, Manuskripte und verschiedene Gefäßformen stellen weitere wichtige Bestandteile der Sammlung dar.

In [141]:
# Häufigste Klassifikation der Bestände innerhalb der mit Bastand größten iran_period Early Islamic
early_islamic_classification = iran_df[iran_df["iran_period"] == "Early Islamic"].groupby("classification")["object_id"].count().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(early_islamic_classification, x="classification", y="object_id", text_auto=",.0f",
             title="Top 10 Classifications in the Early Islamic Period",
             labels={"classification": "Classification", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred",textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False, range=[0, 2000])
fig.show()

# Die Dominanz der Early Islamic Period wird maßgeblich durch Stuckobjekte und Keramik getragen.
# Allein die Kategorie Stucco umfasst mit 1735 Objekten nahezu die gesamte Anzahl aller Stuckobjekte des Iran-Datensatzes (1745) 
# und erklärt damit einen wesentlichen Teil des zeitlichen Sammlungsschwerpunkts.

In [142]:
early_islamic_stucco = iran_df[(iran_df["iran_period"] == "Early Islamic") & (iran_df["classification"] == "Stucco")]

fig = px.histogram(early_islamic_stucco, x="begin_year",
                   title="Distribution of Stucco Objects within the Early Islamic Period",
                   labels={"begin_year": " Dating Start (Year)"})

fig.update_traces(marker_color="darkred")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Number of Objects", showgrid=False)
fig.show()

# Von den 1735 Stucco-Objekten der Early Islamic Period werden 1721 auf das Jahrzehnt 800–809 n. Chr datiert.
# Der bereits in Analyse 2 identifizierte und in Analyse 3 bestätigte Sammlungsschwerpunkt um das frühe 9. Jahrhundert n. Chr. 
# lässt sich somit direkt auf die außergewöhnlich hohe Anzahl frühislamischer Stuckobjekte zurückführen.

In [143]:
# Überprüfung, ob Stucco in der Early Islamic Period einen städtischen Schwerpunkt hat
early_islamic_stucco["city"].value_counts()

# Der zentrale Sammlungsschwerpunkt des Iran-DF lässt sich damit gleichzeitig zeitlich (800–809 n. Chr.), materiell (Stucco) und räumlich (Nishapur) verorten.

city
Nishapur    1734
Rayy           1
Name: count, dtype: int64

In [144]:
# Generelle Überprüfung der Spalte city für Early Islamic
early_islamic_df["city"].value_counts()

# Die Dominanz der Early Islamic Period ist damit nicht nur zeitlich, sondern auch räumlich stark konzentriert.
# Von 3836 Objekten dieser Epoche stammen 3584 aus Nishapur.
# Der historische Sammlungsschwerpunkt der Early Islamic Period ist damit unmittelbar 
# mit einem einzelnen urbanen Zentrum des iranisch-persischen Kulturraums verbunden.

city
Nishapur                   3584
unknown                     184
Qasr-e Abu Nasr              33
Gurgan                       12
Rayy                          8
Amul                          5
Kermanshah                    2
Nishapur or Rayy              2
Tabriz                        1
Amul, Hamadan or Zenjan       1
Isfahan                       1
Yastkand                      1
Saqqizabad                    1
Nihavand                      1
Name: count, dtype: int64

In [145]:
# Prozentuale Berechnung der städischen Verteilung
(early_islamic_df["city"].value_counts() / len(early_islamic_df) * 100).round(2)

city
Nishapur                   93.43
unknown                     4.80
Qasr-e Abu Nasr             0.86
Gurgan                      0.31
Rayy                        0.21
Amul                        0.13
Kermanshah                  0.05
Nishapur or Rayy            0.05
Tabriz                      0.03
Amul, Hamadan or Zenjan     0.03
Isfahan                     0.03
Yastkand                    0.03
Saqqizabad                  0.03
Nihavand                    0.03
Name: count, dtype: float64

In [146]:
# Häufigste Materialien im Iran_DF
iran_df["medium"].value_counts()

# Über 2000 verschiedene Materialtypen, wobei ebenfalls viele Schreibvarianten usw. berücksichtigt werden müssen.

medium
Stucco; carved             698
Ceramic                    471
Copper                     466
Bronze                     446
Stucco; carved, painted    377
                          ... 
Gypsum (?); carved           1
Stucco; carved,              1
Stucco; carved; pigment      1
Stucco; carved: painted      1
Stucco, carved               1
Name: count, Length: 2051, dtype: int64

In [147]:
iran_df["medium"].value_counts().head(10) # Ausgabe der 10 Materialien mit dem häufigsten Vorkommen

medium
Stucco; carved                               698
Ceramic                                      471
Copper                                       466
Bronze                                       446
Stucco; carved, painted                      377
Silver                                       289
Un-baked clay                                199
Stucco; carved; painted                      146
Ink, opaque watercolor, and gold on paper    120
Gold                                         116
Name: count, dtype: int64

Das Attribut medium umfasst insgesamt 2.051 verschiedene Material- und Technikangaben. Eine weitergehende Harmonisierung würde einen erheblichen Bereinigungsaufwand erfordern und steht nur einem begrenzten analytischen Mehrwert gegenüber. Dennoch lassen sich anhand der häufigsten Materialangaben grundlegende Trends erkennen: Die Bestände werden vor allem von Stuck-, Keramik- und Metallobjekten geprägt.

In [148]:
# Erwerbungsarten der Objekte im Iran_DF
iran_df["acquisition_type"].value_counts()

acquisition_type
Other       5973
Gift        1460
Bequest      916
Purchase     744
Name: count, dtype: int64

In [149]:
# Darstellung der Erwerbungsarten als Pie-Diagramm
acquisition_df = iran_df.groupby("acquisition_type")["object_id"].count().sort_values(ascending=False).reset_index()

fig = px.pie(acquisition_df, names="acquisition_type", values="object_id",
             title="Acquisition Types of Iranian-Persian Objects")

fig.update_traces(textinfo="percent+label")
fig.update_layout(title_x=0.5,template="plotly_white")
fig.show()

# Die Erwerbungsgeschichte der iranisch-persischen Bestände wird von der Sammelkategorie "Other" dominiert, auf die knapp zwei Drittel der Objekte entfallen.
# Unter den eindeutig identifizierbaren Erwerbungsarten stellen Schenkungen (16,1 %) die wichtigste Kategorie dar, gefolgt von Vermächtnissen (10,1 %) und Ankäufen (8,2 %).

# Im Vergleich zur Gesamtsammlung werden die iranisch-persischen Objekte wesentlich stärker vom Erwerbungsweg "Other" geprägt.
# Schenkungen spielen eine deutlich geringere Rolle als im Gesamtbestand bzw. Datensatz des Museums.

In [150]:
# Anzeige der zahlenmäßig größten Klassifikation pro iran_period
classification_period_df = iran_df.groupby(["iran_period", "classification"])["object_id"].count().reset_index()

dominant_classification_df = (classification_period_df.sort_values(["iran_period", "object_id"], ascending=[True, False]).drop_duplicates("iran_period").rename(
    columns={"iran_period": "period", "classification": "classification", "object_id": "object_count"}))

dominant_classification_df = dominant_classification_df.set_index("period")

dominant_classification_df = dominant_classification_df.loc[
    [
        "Prehistoric",
        "Pre-Achaemenid",
        "Achaemenid",
        "Hellenistic",
        "Parthian",
        "Sasanian",
        "Early Islamic",
        "Seljuk",
        "Ilkhanid-Timurid",
        "Safavid",
        "Afsharid-Zand",
        "Qajar",
        "Pahlavi",
        "Islamic Republic"
    ]
]

dominant_classification_df = dominant_classification_df.reset_index()

# Gesamtzahl je Periode
period_totals = (iran_df.groupby("iran_period")["object_id"].count().reset_index())

period_totals = period_totals.rename(columns={"iran_period": "period","object_id": "period_total"})

# Merge der Gesamtzahlen
dominant_classification_df = dominant_classification_df.merge(period_totals, on="period")

# Anteil der dominierenden Klassifikation innerhalb der jeweiligen Periode
dominant_classification_df["classification_share"] = (dominant_classification_df["object_count"]/ dominant_classification_df["period_total"] * 100).round(2)

dominant_classification_df # Anzeige

,period,classification,object_count,period_total,classification_share
0,Prehistoric,Ceramics-Vessels,142,295,48.14
1,Pre-Achaemenid,Metalwork-Ornaments,233,930,25.05
2,Achaemenid,Clay-Tablets-Inscribed,56,126,44.44
3,Hellenistic,Ceramics-Vessels,7,22,31.82
4,Parthian,Clay-Sealings,199,1105,18.01
5,Sasanian,Coins,99,274,36.13
6,Early Islamic,Stucco,1735,3836,45.23
7,Seljuk,Ceramics,356,660,53.94
8,Ilkhanid-Timurid,Codices,202,451,44.79
9,Safavid,Codices,195,901,21.64


Die einzelnen historischen Epochen werden durch unterschiedliche Objektgruppen repräsentiert. Besonders auffällig ist die Early Islamic Period, in der Stucco mit 1735 Objekten fast die Hälfte aller Objekte dieser Epoche (45,23 %) ausmacht und damit den zentralen Sammlungsschwerpunkt wesentlich prägt. Die Parther werden dagegen vor allem durch Clay-Sealings (18,01 %), die Sasaniden durch Münzen (36,13 %) und die Safawiden sowie Qajaren durch Handschriften (Codices) repräsentiert. Die zeitlichen Sammlungsschwerpunkte spiegeln damit unterschiedliche materielle Überlieferungsformen innerhalb der iranisch-persischen Geschichte wider.

In [151]:
# Auswerung des Erwerbungsjahres, sofern bekannt
iran_df["acquisition_year"].value_counts()

acquisition_year
1948    1982
1939     799
1940     702
1936     407
1938     387
        ... 
2007       2
2009       2
2011       2
1895       1
2010       1
Name: count, Length: 126, dtype: int64

In [152]:
# Anzeige der unknown-Werte (siehe Datenbereinigung)
print(f"Von {(iran_df["acquisition_year"] == "unknown").sum()} Objekten des Iran_DF ist das Erwebungsjahr unbekannt.")

Von 143 Objekten des Iran_DF ist das Erwebungsjahr unbekannt.


In [153]:
# Auswertung (spalte acquisition_year hat dtype str)
acquisition_year_df = iran_df[iran_df["acquisition_year"] != "unknown"] # Ausnahme der unknown-Werte (durch Anzahl Informationsverlust gering)

acquisition_year_df["acquisition_year"] = acquisition_year_df["acquisition_year"].astype(int) # Umwandlung in int

In [154]:
acquisition_year_df["acquisition_year"].describe().round(2) # Statistische Kennwerte, auch zur Kontrolle

# Das älteste bekannte Erwebsjahr war 1879, das jüngste 2013.

count    8950.00
mean     1944.78
std        23.11
min      1879.00
25%      1936.00
50%      1943.00
75%      1953.00
max      2016.00
Name: acquisition_year, dtype: float64

In [155]:
# Darstellung als Histogramm
fig = px.histogram(acquisition_year_df,x="acquisition_year", 
                   title="Distribution of Acquisition Years",
                   labels={"acquisition_year": "Acquisition Year"})

fig.update_traces(marker_color="darkred")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Number of Objects", showgrid=False)
fig.show()

# Die Erwerbung iranisch-persischer Objekte erfolgte nicht kontinuierlich, sondern in mehreren deutlich erkennbaren Erwerbungswellen.
# Bereits in den späten 1930er-Jahren treten hohe Erwerbungszahlen auf, die mit dem Kriegseintritt der USA 1941 zunächst deutlich zurückgehen.
# Nach dem Ende des Zweiten Weltkriegs folgt mit den Jahren 1948 und 1949 der stärkste Erwerbungsschwerpunkt des gesamten Datensatzes mit über 2000 Objekten.
# In der weiteren Pahlavi-Zeit setzt sich der Bestandsaufbau anschließend auf niedrigerem, aber über längere Zeiträume relativ kontinuierlichem Niveau fort.
# Nach der Islamischen Revolution von 1979 werden weiterhin iranisch-persische Objekte erworben, 
# die Erwerbungszahlen liegen jedoch deutlich unter dem Niveau der vorangegangenen Jahrzehnte.

In [156]:
# Darstellung als Scatterplot zur Hervorhebung von Einzeljahren
acquisition_year_count_df = acquisition_year_df.groupby("acquisition_year")["object_id"].count().reset_index()

fig = px.scatter(acquisition_year_count_df, x="acquisition_year", y="object_id", hover_data=["acquisition_year"],
                 title="Number of Acquisitions by Year",
                 labels={"acquisition_year": "Acquisition Year", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", marker_size=8)
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Der Scatterplot bestätigt die bereits im Histogramm erkennbaren Erwerbungswellen.
# Besonders deutlich treten die Jahre 1939 bis 1940 sowie 1948 und 1949 hervor.
# Das Jahr 1948 stellt mit 1982 Objekten den stärksten Einzelzugang des gesamten Datensatzes dar 
# und markiert den Höhepunkt der Erwerbungstätigkeit.

### Ergebnis Analyse 4

Die Analyse zeigt, dass die iranisch-persischen Bestände materiell und funktional von Fragmenten, Keramik, Stuckobjekten, Münzen und Handschriften geprägt werden. Besonders dominant sind die Klassifikationen Stucco und Ceramics, die zusammen mehr als ein Drittel aller Objekte umfassen. Der Erwerbungsweg der iranisch-persischen Bestände weicht deutlich vom allgemeinen Erwerbungsmuster der Gesamtsammlung ab.

Die einzelnen historischen Epochen werden dabei durch unterschiedliche Objektgruppen repräsentiert. Während die Early Islamic Period nahezu von Stucco-Objekten dominiert wird, prägen Clay-Sealings die Partherzeit, Münzen die Sasanidenzeit und Codices die Safawiden- sowie Qajarenzeit. Die zeitlichen Sammlungsschwerpunkte spiegeln somit unterschiedliche materielle Überlieferungsformen innerhalb der iranisch-persischen Geschichte wider.

Besonders auffällig ist, dass sich die außerordentliche Dominanz der Early Islamic Period unmittelbar auf eine sehr große Zahl frühislamischer Stucco-Objekte zurückführen lässt. Von den 1735 Stucco-Objekten dieser Epoche werden 1721 auf das Jahrzehnt 800–809 n. Chr. datiert. Der bereits in den vorhergehenden Analysen identifizierte Sammlungsschwerpunkt um das frühe 9. Jahrhundert n. Chr. wird dadurch eindeutig erklärt. Räumlich konzentriert sich dieser Schwerpunkt besodners auf Nishapur. Von den 3836 Objekten der Early Islamic Period entfallen 3584 auf diese Stadt (93,43 %). Der zentrale Sammlungsschwerpunkt des Iran-Datensatzes lässt sich damit zeitlich (800–809 n. Chr.), materiell (Stucco) und räumlich (Nishapur) eindeutig verorten.

Auch die Erwerbungsgeschichte weist erkennbare Schwerpunktphasen auf. Auffällig sind starke Erwerbungswellen unmittelbar vor dem Kriegseintritt der USA in den Zweiten Weltkrieg sowie der ausgeprägte Erwerbungsschwerpunkt in den Jahren 1948 und 1949. Die anschließende Pahlavi-Zeit ist durch einen langfristigen, aber deutlich moderateren Bestandsaufbau gekennzeichnet.

Insgesamt zeigt die Analyse, dass die iranisch-persischen Bestände nicht nur zeitlich, sondern auch materiell und funktional klar strukturierte Sammlungsschwerpunkte aufweisen. Die Zusammensetzung der Bestände unterscheidet sich dabei deutlich zwischen den einzelnen historischen Epochen. 

## Analyse 5: Institutionelle Schwerpunkte und Muster

Ziel: Untersuchung institutioneller Sammlungsschwerpunkte und Identifikation struktureller Muster innerhalb der iranisch-persischen Bestände. 

In [157]:
# Nochmalige Anzeige der Verteilung der Objekte des Iran_DF auf die Departments
iran_df["department"].value_counts()

department
Islamic Art                               6187
Ancient Near Eastern Art                  2609
Arms and Armor                             176
Musical Instruments                         45
Costume Institute                           42
Robert Lehman Collection                    15
Greek and Roman Art                          7
American Decorative Arts                     6
The Cloisters                                3
European Sculpture and Decorative Arts       2
Medieval Art                                 1
Name: count, dtype: int64

In [158]:
# Prüfung, inwiefern eine Korrelationsmatrix sinnvoll sein kann
iran_df.select_dtypes(include="number").corr()

# Aufgrund der beschänkten numerischen Werte (und weil begin_centruy direkt aus begin_year abgeleitet wurde) ist eine Korreltationsmatrix für die Analyse nicht sinnvoll.

,object_id,begin_year,begin_century
object_id,1.000000,0.284579,0.276578
begin_year,0.284579,1.000000,0.999583
begin_century,0.276578,0.999583,1.000000


In [159]:
# Bar-Diagramm zur Darstellung der Objekt-Verteilung auf die Departments
department_df = iran_df.groupby("department")["object_id"].count().sort_values(ascending=False).reset_index()

fig = px.bar(department_df, x="department", y="object_id", text_auto=",.0f",
             title="Iranian-Persian Objects by Department",
             labels={"department": "Department", "object_id": "Number of Objects"})

fig.update_traces(marker_color="darkred", textposition="outside")
fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False, range=[0, 7000])
fig.show()

# Das Diagramm bestätigt eine starke Konzentration der iranisch-persischen Bestände auf zwei Museumsabteilungen.
# Allein die Departments Islamic Art (6187 Objekte) und Ancient Near Eastern Art (2609 Objekte) verwalten zusammen einen Großteil aller Objekte des Iran-Datensatzes.
# Andere Museumsabteilungen spielen für die iranisch-persischen Bestände dagegen nur eine untergeordnete Rolle.

In [160]:
#Berechnung des prozentualen Anteils der Departments Islamic Art und Ancient Near Eastern Art
major_departments_df = iran_df[iran_df["department"].isin(["Islamic Art", "Ancient Near Eastern Art"])]

print(f"Die beiden Departments Departments Islamic Art und Ancient Near Eastern Art vereinen {round(len(major_departments_df)/ len(iran_df)* 100,2)} % der Objekte des Iran_DF.")

Die beiden Departments Departments Islamic Art und Ancient Near Eastern Art vereinen 96.73 % der Objekte des Iran_DF.


In [161]:
# Bestimmung der Sammlungsschwerpunkte der beiden dominierenden Departments
# Top 10 Klassifikationen bestimmen
top_10_classifications = major_departments_df["classification"].value_counts().head(10).index

# Alle übrigen Klassifikationen zu "Other" zusammenfassen
major_departments_df.loc[major_departments_df["classification"].isin(top_10_classifications) == False, "classification"] = "Other"

# Gruppierung
department_classification_df = major_departments_df.groupby(["department", "classification"])["object_id"].count().reset_index()

# Alphabetische Sortierung der Legende
department_classification_df = department_classification_df.sort_values("classification")

# Gestapeltes Bar-Diagramm
fig = px.bar(department_classification_df, x="department", y="object_id", color="classification", barmode="stack",
             title="Classifications by Department",
             labels={"department": "Department", "object_id": "Number of Objects", "classification": "Classification"},
             category_orders={
        "department": [
            "Islamic Art",
            "Ancient Near Eastern Art",
            "Arms and Armor"
        ], "classification": sorted(department_classification_df["classification"].unique())})

fig.update_layout(title_x=0.5, template="plotly_white", bargap=0.10)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 6500])
fig.show()

Die weitere Analyse konzentriert sich auf die beiden dominierenden Departments Islamic Art und Ancient Near Eastern Art.

Die beiden Departments weisen deutlich unterschiedliche Sammlungsschwerpunkte auf. Während Islamic Art durch mehrere der zehn häufigsten Klassifikationen geprägt wird, entfällt im Department Ancient Near Eastern Art ein großer Teil der Bestände auf die Sammelkategorie "Other". Dies deutet darauf hin, dass die Bestände des Department Ancient Near Eastern Art stärker über zahlreiche seltenere Klassifikationen verteilt sind, während sich Islamic Art stärker auf einige zentrale Objektgruppen konzentriert.

In [162]:
# Vergleich der zeitlichen Sammlungsschwerpunkten des Department Islamic Art insgesamt und im Gegensatz zu den dort gesammteln Objekten des Iran_DF
islamic_art_total = analysis_df[analysis_df["department"] == "Islamic Art"][["begin_year", "object_id", "classification"]]

islamic_art_total["dataset"] = "Total"

islamic_art_iran = major_departments_df[major_departments_df["department"] == "Islamic Art"][["begin_year", "object_id", "classification"]]

islamic_art_iran["dataset"] = "Iran"

islamic_art_box_df = pd.concat([islamic_art_total, islamic_art_iran])

fig = px.box(islamic_art_box_df, x="dataset", y="begin_year", color="dataset", hover_data=["object_id", "classification"],
             title="Department Islamic Art: Total Collection vs. Iranian-Persian Objects",
             labels={"dataset": "Dataset", "begin_year": "Dating Start (Year)"})

fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Der Median der iranisch-persischen Bestände im Department Islamic Art (800 n. Chr.) liegt unter dem Median des Gesamtbestands der Abteilung (1000 n. Chr.).
# Gleichzeitig weist die Gesamtsammlung des Departments insbesondere in Q3 einen größeren zeitlichen Umfang auf als die iranisch-persischen Bestände.
# Auffällig ist zudem ein einzelner iranischer Ausreißer mit einer Datierung um 5000 v. Chr., 
# der zeitlich deutlich außerhalb des ansonsten überwiegend islamzeitlichen Bestands liegt.

In [163]:
# Vergleich der zeitlichen Sammlungsschwerpunkten des Department Ancient Neat Eastern Art insgesamt und im Gegensatz zu den dort gesammteln Objekten des Iran_DF
ane_total = analysis_df[analysis_df["department"] == "Ancient Near Eastern Art"][["begin_year", "object_id", "classification"]]

ane_total["dataset"] = "Total"

ane_iran = major_departments_df[major_departments_df["department"] == "Ancient Near Eastern Art"][["begin_year", "object_id", "classification"]]

ane_iran["dataset"] = "Iran"

ane_box_df = pd.concat([ane_total, ane_iran])

fig = px.box(ane_box_df, x="dataset", y="begin_year", color="dataset", hover_data=["object_id", "classification"],
             title="Department Ancient Near Eastern Art: Total Collection vs. Iranian-Persian Objects",
             labels={"dataset": "Dataset", "begin_year": "Dating Start (Year)"})

fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Der Iran-Bestand des Department Ancient Near Eastern Art weist insgesamt eine etwas jüngere Verteilung auf als der Gesamtbestand der Abteilung.
# Während der Median des Gesamtbestands bei etwa 900 v. Chr. liegt, verschiebt sich der Median im Iran-Bestand auf 487 v. Chr.
# Gleichzeitig fällt auf, dass die iranisch-persischen Bestände einen größeren Anteil späterer Objekte umfassen, 
# während die sehr frühen Datierungen des Gesamtbestands nur teilweise vertreten sind.
# Die zeitlichen Schwerpunkte des iranischen Teilbestands unterscheiden sich damit stärker vom Gesamtbestand als dies zuvor beim Department Islamic Art zu beobachten war.

In [164]:
# Liniendiagramm zur Untersuchung der zeilichen Sammlungsschwerpunkt der beiden Departments nach iran_period
department_period_df = major_departments_df.groupby(["department", "iran_period"])["object_id"].count().reset_index()

ordered_periods = [
    "Prehistoric",
    "Pre-Achaemenid",
    "Achaemenid",
    "Hellenistic",
    "Parthian",
    "Sasanian",
    "Early Islamic",
    "Seljuk",
    "Ilkhanid-Timurid",
    "Safavid",
    "Afsharid-Zand",
    "Qajar",
    "Pahlavi",
    "Islamic Republic"
]

department_period_df = department_period_df.set_index("iran_period").loc[ordered_periods].reset_index()


fig = px.line(department_period_df, x="iran_period", y="object_id", color="department", markers=True,
              title="Historical Periods by Department",
              labels={"iran_period": "Historical Period", "object_id": "Number of Objects", "department": "Department"})

fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(categoryorder="array", categoryarray=ordered_periods, showgrid=False, tickangle=-45)
fig.update_yaxes(showgrid=False)
fig.show()

# Die beiden Iran-dominierenden Museumsabteilungen weisen deutlich unterschiedliche historische Schwerpunkte auf.
# Während Ancient Near Eastern Art vor allem vorislamische Epochen wie die Pre-Achaemenid- und Partherzeit umfasst, 
# konzentriert sich Islamic Art nahezu vollständig auf die islamischen und frühneuzeitlichen Perioden des iranisch-persischen Kulturraums.
# Die institutionelle Struktur des Museums folgt damit weitgehend einer Trennung zwischen vorislamischen und islamischen Beständen.

In [165]:
# Untersuchung der Erwerbungsjahre in den beiden dominierenden Departments
# Department Islamic Art
acquisition_year_df[acquisition_year_df["department"] == "Islamic Art"]["acquisition_year"].describe().round(2)

count    6073.00
mean     1944.14
std        21.76
min      1879.00
25%      1939.00
50%      1948.00
75%      1948.00
max      2016.00
Name: acquisition_year, dtype: float64

In [166]:
# Department Ancient Near Eastern Art
acquisition_year_df[acquisition_year_df["department"] == "Ancient Near Eastern Art"]["acquisition_year"].describe().round(2)

count    2590.00
mean     1947.59
std        25.08
min      1879.00
25%      1934.00
50%      1947.00
75%      1963.00
max      2006.00
Name: acquisition_year, dtype: float64

Die Erwerbungsjahre der beiden dominierenden Departments weisen ähnliche Medianwerte auf und konzentrieren sich beide auf die Mitte des 20. Jahrhunderts. Während die Erwerbungen des Department Islamic Art stärker um die 1940er-Jahre gebündelt erscheinen, zeigt das Department Ancient Near Eastern Art eine größere zeitliche Streuung und reicht insbesondere im oberen Quartil deutlich weiter in die zweite Hälfte des 20. Jahrhunderts hinein.

In [167]:
# Vergleichende Darstellung der Erwerbungsjahre der beiden führenden Iran-Departments
department_acquisition_df = acquisition_year_df[acquisition_year_df["department"].isin(["Islamic Art", "Ancient Near Eastern Art"])]

fig = px.histogram(department_acquisition_df, x="acquisition_year", color="department", barmode="overlay", opacity=0.7,
                   title="Distribution of Acquisition Years by Department", 
                   labels={"acquisition_year": "Acquisition Year", "department": "Department"})

fig.update_layout(title_x=0.5, template="plotly_white")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Number of Objects", showgrid=False)
fig.show()

# Beide Departments weisen bereits in den späten 1930er-Jahren deutliche Erwerbungswellen auf.
# Die stärksten Erwerbungsspitzen entfallen jedoch auf das Department Islamic Art, insbesondere in den Jahren 1938–1940 sowie 1948–1949.
# Ancient Near Eastern Art erreicht seinen Höhepunkt ebenfalls kurz vor dem Kriegseintritt der USA, weist anschließend jedoch deutlich geringere Ausschläge auf.
# Während die außergewöhnlichen Erwerbungswellen der 1940er-Jahre überwiegend durch Islamic Art geprägt werden, 
# trägt Ancient Near Eastern Art stärker zum kontinuierlichen Bestandsaufbau der folgenden Jahrzehnte bei.

### Ergebnis Analyse 5

Die iranisch-persischen Bestände konzentrieren sich institutionell nahezu vollständig auf die Departments Islamic Art und Ancient Near Eastern Art, die zusammen rund 97 % des gesamten Iran-Datensatzes verwalten.

Beide Abteilungen weisen deutlich unterschiedliche Sammlungsschwerpunkte auf. Während Islamic Art vor allem islamische und frühneuzeitliche Bestände umfasst, konzentriert sich Ancient Near Eastern Art auf die vorislamischen Epochen des iranischen Kulturraums. Diese Trennung zeigt sich sowohl in der zeitlichen Verteilung der Objekte als auch in den dominierenden Klassifikationen der jeweiligen Departments.

Auch die Erwerbungsgeschichte unterscheidet sich zwischen beiden Abteilungen. Die großen Erwerbungswellen unmittelbar vor dem Kriegseintritt der USA in den Zweiten Weltkrieg sowie der ausgeprägte Erwerbungsschwerpunkt der Jahre 1948 und 1949 gehen überwiegend auf das Department Islamic Art zurück. Ancient Near Eastern Art weist dagegen einen gleichmäßigeren Bestandsaufbau auf und prägt insbesondere die weiteren Zugänge während der Pahlavi-Zeit.

Die institutionelle Organisation des Metropolitan Museum of Art bildet damit die historische Gliederung des iranisch-persischen Kulturraums weitgehend ab und trennt die Bestände im Wesentlichen entlang der Grenze zwischen vorislamischer und islamischer Geschichte.

## Schritt 6: Daten visualisieren & präsentieren

### Übergeordnete Fragestellung

Welche kulturellen, geografischen und zeitlichen Schwerpunkte prägen die Sammlungsbestände des Metropolitan Museum of Art und welche Merkmale kennzeichnen die iranischen Bestände innerhalb dieser Sammlung?

Die Sammlungsbestände des Metropolitan Museum of Art zeichnen sich durch eine außergewöhnlich hohe kulturelle, geografische und zeitliche Vielfalt aus. Gleichzeitig zeigen die Metadaten erhebliche Dokumentationslücken, insbesondere bei Kultur- und Herkunftsangaben. Die Sammlung konzentriert sich zeitlich vor allem auf die Neuzeit und das frühe 20. Jahrhundert.

Die iranisch-persischen Bestände nehmen innerhalb dieser Gesamtstruktur zwar nur einen kleinen Anteil ein, bilden jedoch einen historisch klar abgegrenzten und analytisch besonders aussagekräftigen Teilbestand. Sie konzentrieren sich auf wenige dominante historische Epochen, werden von charakteristischen Objektgruppen geprägt und spiegeln institutionell die Trennung zwischen vorislamischer und islamischer Geschichte wider.

#### Leitfrage 1: Welche kulturellen, geografischen und zeitlichen Schwerpunkte lassen sich in den Sammlungsbeständen des Metropolitan Museum of Art erkennen?

Die Sammlung weist mehr als 7.000 Kulturangaben und über 1.000 Herkunftsangaben auf und besitzt damit eine außergewöhnlich hohe kulturelle und geografische Diversität. Unter den dokumentierten Kulturangaben dominieren insbesondere amerikanische, französische, japanische und chinesische Objekte. Bei den Herkunftsländern treten vor allem Ägypten, die Vereinigten Staaten und Iran hervor.

Gleichzeitig zeigen die Metadaten erhebliche Dokumentationsprobleme. Für rund 58 % der Objekte fehlt eine Kulturangabe, für etwa 83 % eine Herkunftsangabe. Die geographischen Schwerpunkte können daher nur auf Basis des dokumentierten Teilbestands bestimmt werden.

Zeitlich reicht die Sammlung von der Vor- und Frühgeschichte bis in die Gegenwart. Die höchste Objektdichte liegt jedoch eindeutig auf der Neuzeit, insbesondere auf dem 18. bis frühen 20. Jahrhundert.

#### Leitfrage 2: Welche Rolle spielen die iranischen Bestände innerhalb der Sammlung des Metropolitan Museum of Art?

Der definierte Iran-Datensatz umfasst 9.093 Objekte und entspricht rund 2 % der Gesamtbestände des Museums. Trotz dieses vergleichsweise kleinen Anteils bildet er einen außergewöhnlich großen und historisch breit angelegten Teilbestand.

Die Konstruktion des Datensatzes erforderte die Kombination geografischer (country_clean, region) und kulturhistorischer (culture) Selektionskriterien. Dabei zeigte sich, dass zahlreiche für die Fallstudie relevante Objekte nicht über die Herkunftsangaben, sondern erst über regionale und kulturelle Attribute identifiziert werden konnten.

Besonders auffällig ist die starke Präsenz iranisch-persischer Objekte im frühen 9. Jahrhundert n. Chr. Bereits der Vergleich mit der Gesamtsammlung weist in diesem Zeitraum auf einen eigenständigen historischen Sammlungsschwerpunkt hin.

Von den 9.093 Objekten stammen 8798 Objekte (96,76 %) aus der Zeit nach 1000 v. Chr. und damit aus dem engeren Zeitraum der iranisch-persischen Geschichte. Der Datensatz stellt somit eine große und zugleich fachlich nachvollziehbar abgegrenzte Untersuchungsgrundlage dar.

#### Leitfrage 3: Welche zeitlichen, materiellen und institutionellen Schwerpunkte weisen die iranischen Bestände auf?

Die zeitliche Analyse zeigt eine starke Konzentration auf wenige historische Kernperioden. Die Early Islamic Period umfasst allein 3836 Objekte bzw. 42,19 % des gesamten Iran-Datensatzes. Daneben sind insbesondere die Parther-, vorachämenidischen und safawidischen Perioden stark vertreten.

Innerhalb der Early Islamic Period konzentrieren sich die Bestände außergewöhnlich stark auf das Jahrzehnt 800–809 n. Chr. mit 2525 Objekten. Dieser Schwerpunkt prägt die gesamte zeitliche Struktur des Iran-Datensatzes.

Materiell und funktional werden die Bestände von Fragmenten, Keramik, Stuckobjekten, Münzen und Handschriften dominiert. Besonders hervorzuheben ist die Klassifikation Stucco, die mit 1745 Objekten die größte Einzelgruppe darstellt. Von den 1735 Stucco-Objekten der Early Islamic Period werden 1721 auf das Jahrzehnt 800–809 n. Chr. datiert. Darüber hinaus konzentriert sich die Early Islamic Period auch räumlich stark auf Nishapur. Von den 3836 Objekten dieser Epoche entfallen 3584 auf diese Stadt (93,43 %). Die zeitliche, materielle und räumliche Konzentration der Bestände fällt damit in bemerkenswerter Weise zusammen.

Die einzelnen historischen Epochen werden durch unterschiedliche materielle Traditionen repräsentiert. Während die Early Islamic Period von Stucco-Objekten geprägt wird, dominieren Clay-Sealings die Partherzeit, Münzen die Sasanidenzeit und Codices die Safawiden- sowie Qajarenzeit.

Institutionell konzentrieren sich die Bestände nahezu vollständig auf die beiden Departments Islamic Art und Ancient Near Eastern Art, die zusammen rund 97 % aller iranisch-persischen Objekte verwalten. Dabei zeigt sich eine klare historische Arbeitsteilung:

Ancient Near Eastern Art konzentriert sich auf die vorislamischen Epochen. Islamic Art umfasst vor allem die islamischen und frühneuzeitlichen Perioden.

Auch die Erwerbungsgeschichte weist deutliche Schwerpunktphasen auf. Starke Erwerbungswellen treten unmittelbar vor dem Kriegseintritt der USA in den Zweiten Weltkrieg sowie in den Jahren 1948 und 1949 auf. Der anschließende Bestandsaufbau während der Pahlavi-Zeit erfolgt deutlich kontinuierlicher und auf niedrigerem Niveau. Zugleich unterscheidet sich das Erwerbungsmuster der iranisch-persischen Bestände deutlich von den allgemeinen Erwerbungsstrukturen der Gesamtsammlung. Dies ist provinienzhistorisch auch für Leitfrage 2 von Interesse.


### Fazit

Die Untersuchung zeigt, dass die iranisch-persischen Bestände zwar nur einen kleinen Teil der Gesamtbestände des Metropolitan Museum of Art ausmachen, innerhalb dieses Teilbestands jedoch außergewöhnlich klare historische, materielle und institutionelle Strukturen vorliegen.

Besonders prägend sind die Early Islamic Period, die Konzentration auf das frühe 9. Jahrhundert n. Chr. sowie die außerordentlich starke Präsenz frühislamischer Stucco-Objekte.Diese Schwerpunkte lassen sich zudem räumlich vor allem auf Nishapur zurückführen, das innerhalb der Early Islamic Period die mit Abstand wichtigste dokumentierte Stadt des Iran_DF darstellt. Die institutionelle Organisation der Bestände folgt gleichzeitig einer bemerkenswert klaren Trennung zwischen vorislamischer und islamischer Geschichte.

Die Analyse verdeutlicht damit, dass die iranisch-persischen Bestände nicht nur einen quantitativ bedeutenden Teilbestand darstellen, sondern innerhalb der Museumssammlung einen eigenständigen historischen und kulturellen Schwerpunkt bilden.

In [168]:
# Vorbereitung für Streamlit-Dashboard
# analysis_df.to_csv("analysis_df_export.csv", index=False)
# iran_df.to_csv("iran_df_export.csv", index=False)


# Die Analyse bzw. ausgewählte Diagramme sollen als Streamlit-Dashboard dargestellt werden. 
# Hierfür werden analysis_df und iran_df als .csv ausgeleitet, um sie in eienr dashboard.py verwenden zu können.

## Schritt 7: Handlungsempfehlungen

#### Handlungsempfehlung zu Leitfrage 1

Die Qualität der Sammlungsmetadaten sollte insbesondere bei Kultur- und Herkunftsangaben verbessert werden (Nachtrag und künftige Erfassungsstandards), da die hohen Anteile fehlender Werte die Aussagekraft zukünftiger Analysen erheblich einschränken.

#### Handlungsempfehlung zu Leitfrage 2

Iranisch-persische Bestände sollten aufgrund ihrer Größe, historischen Tiefe und klaren Struktur als eigenständiger Forschungsschwerpunkt innerhalb der digitalen Sammlungsanalyse systematisch erschlossen und weiter untersucht werden. Die auffälligen Erwerbungswellen der 1930er- bis 1960er-Jahre sollten in zukünftigen Provenienz- und Sammlungsgeschichtsstudien näher untersucht werden.

#### Handlungsempfehlung zu Leitfrage 3

Die Early Islamic Period und insbesondere die außergewöhnlich große Gruppe frühislamischer Stucco-Objekte sollten als zentraler Sammlungsschwerpunkt in Forschung, Digitalisierung, Vermittlung und zukünftigen Ausstellungen gezielt hervorgehoben werden. 